In [ ]:
print("🔗 Montando Google Drive...")
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 🩺 Detección de Cáncer de Mama — Pipeline Corregido

Este notebook corrige la versión anterior y organiza el proyecto en dos fases claras:

**FASE 1 — EDA (Análisis Exploratorio):** carga y describe los datasets, separa
visualmente ejemplos benignos vs malignos, y guarda una muestra de imágenes para
revisión manual.

**FASE 2 — Modelado:** entrena **5 modelos** sobre los datos ya explorados:
- **3 modelos únicos**: CNN (EfficientNetB4) sobre imágenes, XGBoost sobre datos
  tabulares, Random Forest sobre datos tabulares.
- **2 modelos híbridos**: CNN+Random Forest y CNN+XGBoost (combinan dos algoritmos
  sobre el mismo dataset de imágenes — ver nota en la Fase 2).

**Cambios de esta corrección:**
- `EPOCHS` bajado a **12** (antes 20) para acelerar las primeras pruebas.
- Los splits se calculan explícitamente en porcentajes (80% entrenamiento+validación
  / 20% prueba para imágenes; 80/20 directo para datos tabulares).
- Cada modelo se guarda en **un solo archivo** (los híbridos se empaquetan en un
  `.zip` que contiene el extractor CNN + el clasificador).
- Se guarda una muestra del set de prueba (imágenes físicas + CSV) para revisión
  posterior.
- Las clases (`Config`, `BreastCancerEDA`, `BreastCancerDetectionSystem`) ahora se
  definen **una sola vez** (antes se redefinían en varias celdas, lo que generaba
  versiones "dummy" no entrenadas si se ejecutaban fuera de orden).

In [ ]:
print("📦 Instalando dependencias necesarias...")
!pip install -q opencv-python seaborn matplotlib pandas numpy scikit-learn plotly pillow tqdm
!pip install -q tensorflow keras scikit-image xgboost lightgbm albumentations
!pip install -q openpyxl python-docx

print("📋 Verificando versiones instaladas...")
import sys
print(f"Python: {sys.version}")
print("✅ Instalación de dependencias completada!")


In [ ]:
print("📚 Importando librerías...")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os
import shutil
import glob
import zipfile
import json
import joblib
import warnings
import time
import math
from pathlib import Path
from datetime import datetime
from IPython.display import display
from tqdm import tqdm

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50, EfficientNetB4, DenseNet121

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, roc_auc_score,
    precision_score, recall_score, f1_score, roc_curve
)
import xgboost as xgb
import albumentations as A

# --- Pruebas estadísticas ---
from scipy import stats
from scipy.stats import ttest_rel, wilcoxon, norm, chi2

# --- Reportes profesionales (Excel / PDF / Word) ---
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from matplotlib.backends.backend_pdf import PdfPages
from docx import Document
from docx.shared import Pt, Inches, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH

warnings.filterwarnings('ignore')
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10
%matplotlib inline

print("✅ Todas las librerías importadas correctamente!")


In [ ]:
def verify_drive_structure():
    """Verificar que todos los archivos estén en Drive"""
    print("\n🔍 VERIFICANDO ESTRUCTURA DE ARCHIVOS EN DRIVE")
    print("="*50)

    base_path = "/content/drive/MyDrive/DataSet"

    folders_to_check = [
        "CSVFiles",
        "BreastCancer_Images/jpeg",
        "BreastCancer_Tabular",
        "Database"
    ]
    files_to_check = [
        "CSVFiles/calc_case_description_train_set.csv",
        "CSVFiles/calc_case_description_test_set.csv",
        "CSVFiles/mass_case_description_train_set.csv",
        "CSVFiles/mass_case_description_test_set.csv",
        "CSVFiles/meta.csv",
        "CSVFiles/dicom_info.csv",
        "BreastCancer_Tabular/data.csv"
    ]

    print("📁 Verificando carpetas:")
    for folder in folders_to_check:
        folder_path = os.path.join(base_path, folder)
        print(f"   {'✅' if os.path.exists(folder_path) else '❌'} {folder}")

    print("\n📄 Verificando archivos críticos:")
    for file in files_to_check:
        file_path = os.path.join(base_path, file)
        if os.path.exists(file_path):
            file_size = os.path.getsize(file_path) / 1024
            print(f"   ✅ {file} ({file_size:.1f} KB)")
        else:
            print(f"   ❌ {file} - NO ENCONTRADO")

    images_path = os.path.join(base_path, "BreastCancer_Images/jpeg")
    if os.path.exists(images_path):
        subdirs = [d for d in os.listdir(images_path) if os.path.isdir(os.path.join(images_path, d))]
        print(f"\n🖼️ Subcarpetas de imágenes encontradas: {len(subdirs)}")

    return os.path.exists(base_path)

## 📊 FASE 1: Análisis Exploratorio de Datos (EDA)

Antes de entrenar cualquier modelo, esta fase:
1. Carga los datasets CBIS-DDSM (calcificaciones + masas) y Wisconsin.
2. Describe distribución de patologías y características clínicas (BI-RADS, subtlety).
3. Verifica la disponibilidad real de imágenes en Drive.
4. Construye el índice real de imágenes con su ruta absoluta.
5. **Separa visualmente ejemplos benignos vs malignos** (cuadrícula de muestras).
6. **Guarda una muestra física** de imágenes por clase en `Resultados/muestras/`.

In [ ]:
class Config:
    """Configuración global del proyecto"""

    # Rutas
    BASE_PATH = "/content/drive/MyDrive/DataSet"
    RESULTS_PATH = f"{BASE_PATH}/Resultados"
    MODELS_PATH = f"{BASE_PATH}/Models"

    # Parámetros de imagen
    IMG_SIZE = 224
    IMG_CHANNELS = 3
    BATCH_SIZE = 32

    # Parámetros de entrenamiento (EPOCHS bajado de 20 a 12 para pruebas iniciales)
    EPOCHS = 12
    LEARNING_RATE = 1e-4
    PATIENCE = 6

    # División de datos para IMÁGENES, como % del total:
    #   80% entrenamiento+validación (72% train + 8% val) / 20% prueba
    TRAIN_PCT = 0.72
    VAL_PCT = 0.08
    TEST_PCT = 0.20

    # División de datos TABULARES (Wisconsin): 80% train / 20% test directo
    TABULAR_TEST_SIZE = 0.2

    # Muestras para EDA / revisión manual
    N_SAMPLE_IMAGES_PER_CLASS = 6   # para mostrar en pantalla
    N_TEST_IMAGES_TO_SAVE = 15      # para copiar a disco por clase

    RANDOM_STATE = 42
    MIXED_PRECISION = True

    # --------------------------------------------------------------
    # Nuevos parámetros para Tuning y Validación Cruzada
    # --------------------------------------------------------------
    N_FOLDS = 5                 # Folds para la validación cruzada (5-Fold CV)
    TUNING_CV = 3                # Folds internos usados durante el tuning (Grid/RandomizedSearch)
    N_ITER_RANDOM = 10           # Iteraciones para RandomizedSearchCV (Random Forest)
    FEATURE_BATCH_SIZE = 64      # Tamaño de lote para extracción de features CNN

    # Grid de hiperparámetros - XGBoost tabular
    XGB_PARAM_GRID = {
        'max_depth': [3, 6],
        'learning_rate': [0.05, 0.1],
        'n_estimators': [100, 150]
    }

    # Distribución de hiperparámetros - Random Forest tabular
    RF_PARAM_DIST = {
        'n_estimators': [50, 100, 150, 200],
        'max_depth': [None, 5, 10, 15],
        'min_samples_split': [2, 5, 10]
    }

    # Corrección de Bonferroni para pruebas estadísticas múltiples
    ALPHA = 0.05
    N_COMPARISONS = 10  # C(5,2) pares de modelos
    ALPHA_BONFERRONI = ALPHA / N_COMPARISONS

    # --------------------------------------------------------------
    # Para reportes
    # --------------------------------------------------------------
    REPORTS_PATH = f"{BASE_PATH}/Reports"
    EXCEL_NAME = "resultados_modelos.xlsx"
    PDF_NAME = "reporte_figuras.pdf"
    WORD_NAME = "reporte_interpretativo.docx"


os.makedirs(Config.MODELS_PATH, exist_ok=True)
os.makedirs(Config.RESULTS_PATH, exist_ok=True)
os.makedirs(Config.REPORTS_PATH, exist_ok=True)

print("✅ Config cargado:")
print(f"   • EPOCHS = {Config.EPOCHS}")
print(f"   • Split imágenes -> Train {Config.TRAIN_PCT*100:.0f}% / Val {Config.VAL_PCT*100:.0f}% / Test {Config.TEST_PCT*100:.0f}%")
print(f"   • Split tabular  -> Train {(1-Config.TABULAR_TEST_SIZE)*100:.0f}% / Test {Config.TABULAR_TEST_SIZE*100:.0f}%")
print(f"   • Validación cruzada: {Config.N_FOLDS}-Fold")
print(f"   • Reportes en: {Config.REPORTS_PATH}")


In [ ]:
if tf.config.list_physical_devices('GPU'):
    print("🎮 GPU detectada - Configurando...")
    gpus = tf.config.experimental.list_physical_devices('GPU')
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
        print(f"✅ GPU configurada: {gpus[0]}")
    except RuntimeError as e:
        print(f"⚠️ Error configurando GPU: {e}")
else:
    print("⚠️ No se detectó GPU - Usando CPU (el entrenamiento será considerablemente más lento)")

if Config.MIXED_PRECISION:
    policy = tf.keras.mixed_precision.Policy('mixed_float16')
    tf.keras.mixed_precision.set_global_policy(policy)
    print("✅ Precisión mixta habilitada")

In [ ]:
class BreastCancerEDA:
    """
    Clase para el Análisis Exploratorio de Datos (EDA) del proyecto de
    detección de cáncer de mama. Se encarga de:
      1. Cargar los datasets (CBIS-DDSM + Wisconsin)
      2. Explorar y describir su contenido
      3. Construir el índice de imágenes (image_df) con rutas absolutas
      4. Separar visualmente ejemplos benignos vs malignos
      5. Guardar una muestra de imágenes por clase para revisión manual
    """

    def __init__(self, base_path=Config.BASE_PATH):
        self.base_path = Path(base_path)
        self.csv_path = self.base_path / "CSVFiles"
        self.images_path = self.base_path / "BreastCancer_Images" / "jpeg"
        self.tabular_path = self.base_path / "BreastCancer_Tabular"
        self.database_path = self.base_path / "Database"
        self.results_path = self.base_path / "Resultados"
        self.samples_path = self.results_path / "muestras"

        self.results_path.mkdir(exist_ok=True, parents=True)
        self.samples_path.mkdir(exist_ok=True, parents=True)
        print(f"📂 Carpeta de resultados: {self.results_path}")

        self.image_df = pd.DataFrame()

    # ------------------------------------------------------------------
    def load_all_data(self):
        """Cargar todos los archivos CSV y datos tabulares"""
        print("\n🔄 Cargando todos los datasets...")
        try:
            print("   📋 Cargando calcificaciones...")
            self.calc_train = pd.read_csv(self.csv_path / "calc_case_description_train_set.csv")
            self.calc_test = pd.read_csv(self.csv_path / "calc_case_description_test_set.csv")

            print("   📋 Cargando masas...")
            self.mass_train = pd.read_csv(self.csv_path / "mass_case_description_train_set.csv")
            self.mass_test = pd.read_csv(self.csv_path / "mass_case_description_test_set.csv")

            print("   📋 Cargando metadatos...")
            self.meta = pd.read_csv(self.csv_path / "meta.csv")

            try:
                self.dicom_info = pd.read_csv(self.csv_path / "dicom_info.csv")
            except Exception:
                print("   ⚠️ dicom_info.csv no encontrado (opcional)")
                self.dicom_info = None

            print("   📋 Cargando datos Wisconsin...")
            self.wisconsin_data = pd.read_csv(self.tabular_path / "data.csv")

            self.calc_all = pd.concat([self.calc_train, self.calc_test], ignore_index=True)
            self.mass_all = pd.concat([self.mass_train, self.mass_test], ignore_index=True)

            print("✅ Datos cargados exitosamente!")
            self._print_data_summary()

        except Exception as e:
            print(f"❌ Error cargando datos: {str(e)}")
            raise

    def _print_data_summary(self):
        print("\n📊 RESUMEN DE DATASETS:")
        print(f"📋 Calcificaciones: {len(self.calc_all)} casos")
        print(f"📋 Masas: {len(self.mass_all)} casos")
        print(f"📋 Metadatos: {len(self.meta)} series")
        print(f"📋 Wisconsin (tabular): {len(self.wisconsin_data)} casos")

    def show_data_samples(self):
        print("\n🔍 MUESTRAS DE DATASETS")
        print("="*50)
        print("\n1️⃣ CALCIFICACIONES (primeras 3 filas):")
        print("Columnas:", list(self.calc_all.columns))
        display(self.calc_all.head(3))

        print("\n2️⃣ MASAS (primeras 3 filas):")
        print("Columnas:", list(self.mass_all.columns))
        display(self.mass_all.head(3))

        print("\n3️⃣ WISCONSIN (primeras 3 filas):")
        print("Columnas:", list(self.wisconsin_data.columns))
        display(self.wisconsin_data.head(3))

        print("\n4️⃣ METADATOS (primeras 3 filas):")
        print("Columnas:", list(self.meta.columns))
        display(self.meta.head(3))

    def analyze_pathology_distribution(self):
        print("\n🔍 ANÁLISIS DE PATOLOGÍAS")
        print("="*50)

        if 'pathology' in self.calc_all.columns:
            calc_pathology = self.calc_all['pathology'].value_counts()
            print("\n📈 CALCIFICACIONES:")
            for path, count in calc_pathology.items():
                pct = (count/len(self.calc_all))*100
                print(f"   {path}: {count} ({pct:.1f}%)")

        if 'pathology' in self.mass_all.columns:
            mass_pathology = self.mass_all['pathology'].value_counts()
            print("\n📈 MASAS:")
            for path, count in mass_pathology.items():
                pct = (count/len(self.mass_all))*100
                print(f"   {path}: {count} ({pct:.1f}%)")

        if 'diagnosis' in self.wisconsin_data.columns:
            wis_diagnosis = self.wisconsin_data['diagnosis'].value_counts()
            print("\n📈 WISCONSIN:")
            for diag, count in wis_diagnosis.items():
                pct = (count/len(self.wisconsin_data))*100
                label = "MALIGNANT" if diag == 'M' else "BENIGN"
                print(f"   {label} ({diag}): {count} ({pct:.1f}%)")

    def create_distribution_plots(self):
        print("\n📊 Creando gráficos de distribución...")
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        fig.suptitle('DISTRIBUCIÓN DE PATOLOGÍAS', fontsize=16, fontweight='bold')

        if 'pathology' in self.calc_all.columns:
            calc_counts = self.calc_all['pathology'].value_counts()
            axes[0].pie(calc_counts.values, labels=calc_counts.index, autopct='%1.1f%%', startangle=90)
            axes[0].set_title(f'Calcificaciones\n(n={len(self.calc_all)})')

        if 'pathology' in self.mass_all.columns:
            mass_counts = self.mass_all['pathology'].value_counts()
            axes[1].pie(mass_counts.values, labels=mass_counts.index, autopct='%1.1f%%', startangle=90)
            axes[1].set_title(f'Masas\n(n={len(self.mass_all)})')

        if 'diagnosis' in self.wisconsin_data.columns:
            wis_counts = self.wisconsin_data['diagnosis'].value_counts()
            labels = ['MALIGNANT' if x == 'M' else 'BENIGN' for x in wis_counts.index]
            axes[2].pie(wis_counts.values, labels=labels, autopct='%1.1f%%', startangle=90)
            axes[2].set_title(f'Wisconsin\n(n={len(self.wisconsin_data)})')

        plt.tight_layout()
        plt.savefig(self.results_path / 'pathology_distribution.png', dpi=300, bbox_inches='tight')
        plt.show()

    def analyze_clinical_features(self):
        print("\n🔍 ANÁLISIS DE CARACTERÍSTICAS CLÍNICAS")
        print("="*50)

        if 'assessment' in self.calc_all.columns:
            print("\n🏥 BI-RADS Assessment - Calcificaciones:")
            for assess, count in self.calc_all['assessment'].value_counts().sort_index().items():
                print(f"   Assessment {assess}: {count}")

        if 'assessment' in self.mass_all.columns:
            print("\n🏥 BI-RADS Assessment - Masas:")
            for assess, count in self.mass_all['assessment'].value_counts().sort_index().items():
                print(f"   Assessment {assess}: {count}")

        if 'subtlety' in self.calc_all.columns:
            print("\n👁️ Subtlety - Calcificaciones:")
            for sub, count in self.calc_all['subtlety'].value_counts().sort_index().items():
                print(f"   Subtlety {sub}: {count}")

        if 'subtlety' in self.mass_all.columns:
            print("\n👁️ Subtlety - Masas:")
            for sub, count in self.mass_all['subtlety'].value_counts().sort_index().items():
                print(f"   Subtlety {sub}: {count}")

    def map_images_to_metadata(self):
        print("\n🔗 MAPEANDO IMÁGENES CON METADATOS")
        print("="*50)

        if 'SeriesInstanceUID' in self.meta.columns:
            series_in_meta = set(self.meta['SeriesInstanceUID'].unique())
            print(f"📋 Series en meta.csv: {len(series_in_meta)}")
        else:
            print("❌ No se encontró columna SeriesInstanceUID en metadatos")
            return 0

        if self.images_path.exists():
            image_folders = [d.name for d in self.images_path.iterdir() if d.is_dir()]
            series_with_images = set(image_folders)
            print(f"📁 Carpetas de imágenes: {len(series_with_images)}")

            matching_series = series_in_meta.intersection(series_with_images)
            missing_images = series_in_meta - series_with_images
            extra_images = series_with_images - series_in_meta

            print("\n🔗 CORRESPONDENCIAS:")
            print(f"   ✅ Series con imágenes y metadatos: {len(matching_series)}")
            print(f"   ❌ Series en meta.csv sin imágenes: {len(missing_images)}")
            print(f"   ⚠️ Carpetas de imágenes sin metadatos: {len(extra_images)}")

            return len(matching_series)
        else:
            print("❌ Carpeta de imágenes no existe")
            return 0

    def check_image_files(self, sample_size=5):
        print("\n🖼️ VERIFICANDO ARCHIVOS DE IMAGEN")
        print("="*50)

        if not self.images_path.exists():
            print("❌ Carpeta de imágenes no existe")
            print(f"💡 Ruta esperada: {self.images_path}")
            return 0

        subdirs = [d for d in self.images_path.iterdir() if d.is_dir()]
        print(f"📁 Subcarpetas encontradas: {len(subdirs)}")

        image_extensions = ['*.jpg', '*.jpeg', '*.png']
        all_images = []
        for ext in image_extensions:
            found = list(self.images_path.rglob(ext))
            all_images.extend(found)
            if found:
                print(f"   {ext}: {len(found)} archivos")

        print(f"\n📷 Total archivos de imagen encontrados: {len(all_images)}")

        if len(all_images) > 0:
            for i, img_file in enumerate(all_images[:sample_size]):
                img = cv2.imread(str(img_file), cv2.IMREAD_GRAYSCALE)
                if img is not None:
                    h, w = img.shape
                    print(f"   {i+1}. {img_file.name}: {w}x{h}")

        return len(all_images)

    # ------------------------------------------------------------------
    # Índice de imágenes (CORREGIDO: empareja por carpeta SeriesInstanceUID,
    # ya que los archivos dentro pierden el nombre original del .dcm)
    # ------------------------------------------------------------------
    def build_image_index(self):
        """Construye self.image_df combinando calcificaciones + masas, con
        la ruta absoluta real de cada imagen en Drive."""
        print("\n🖼️ CONSTRUYENDO ÍNDICE DE IMÁGENES (benignas/malignas)")
        print("="*50)

        all_cases = []
        for _, row in self.calc_all.iterrows():
            if pd.notna(row['image file path']):
                all_cases.append({
                    'patient_id': row['patient_id'],
                    'image_path': row['image file path'],
                    'pathology': row['pathology'],
                    'type': 'calcification',
                    'assessment': row['assessment'],
                    'subtlety': row['subtlety']
                })
        for _, row in self.mass_all.iterrows():
            if pd.notna(row['image file path']):
                all_cases.append({
                    'patient_id': row['patient_id'],
                    'image_path': row['image file path'],
                    'pathology': row['pathology'],
                    'type': 'mass',
                    'assessment': row['assessment'],
                    'subtlety': row['subtlety']
                })

        image_df = pd.DataFrame(all_cases)
        image_df = image_df[image_df['pathology'].isin(['MALIGNANT', 'BENIGN'])].reset_index(drop=True)
        image_df['label'] = (image_df['pathology'] == 'MALIGNANT').astype(int)

        print(f"📊 Casos encontrados en CSV (antes de mapeo): {len(image_df)}")

        series_to_path = {}
        total_images_indexed = 0
        if self.images_path.exists():
            for series_folder in self.images_path.iterdir():
                if series_folder.is_dir():
                    imgs = []
                    for ext in ['*.jpg', '*.jpeg', '*.png']:
                        imgs.extend(series_folder.glob(ext))
                    if imgs:
                        series_to_path[series_folder.name] = str(sorted(imgs)[0])
                        total_images_indexed += len(imgs)

        print(f"✅ {total_images_indexed} imágenes indexadas en {len(series_to_path)} carpetas (series).")
        self.series_to_path = series_to_path

        def get_full_path(rel_path):
            parts = Path(rel_path).parts
            if len(parts) < 2:
                return None
            series_id = parts[-2]
            if series_id in series_to_path:
                return series_to_path[series_id]
            for key in series_to_path:
                if series_id in key or key in series_id:
                    return series_to_path[key]
            return None

        print("📌 Mapeando rutas de imágenes (por SeriesInstanceUID)...")
        image_df['full_path'] = image_df['image_path'].apply(get_full_path)
        image_df = image_df[image_df['full_path'].notna()].reset_index(drop=True)

        self.image_df = image_df

        n_total = len(image_df)
        n_mal = (image_df['label'] == 1).sum()
        n_ben = (image_df['label'] == 0).sum()
        print("\n📊 Dataset de imágenes indexado:")
        print(f"   • Total casos con imagen: {n_total}")
        if n_total > 0:
            print(f"   • Malignos: {n_mal} ({n_mal/n_total*100:.1f}%)")
            print(f"   • Benignos: {n_ben} ({n_ben/n_total*100:.1f}%)")
        print(f"   • Calcificaciones: {(image_df['type']=='calcification').sum()}")
        print(f"   • Masas: {(image_df['type']=='mass').sum()}")

        if n_total == 0:
            print("\n⚠️ No se encontraron imágenes. Revisa la estructura de carpetas en Drive.")

        return self.image_df

    def show_sample_images_by_class(self, n=Config.N_SAMPLE_IMAGES_PER_CLASS):
        """Separa visualmente ejemplos BENIGNOS vs MALIGNOS: muestra una
        cuadrícula con n imágenes de cada clase."""
        if self.image_df.empty:
            print("⚠️ Ejecuta build_image_index() primero.")
            return

        print(f"\n🖼️ EJEMPLOS POR CLASE (benignas vs malignas), n={n} por clase")
        print("="*50)

        n_ben_avail = (self.image_df['label'] == 0).sum()
        n_mal_avail = (self.image_df['label'] == 1).sum()
        n = min(n, n_ben_avail, n_mal_avail) if min(n_ben_avail, n_mal_avail) > 0 else 0
        if n == 0:
            print("⚠️ No hay suficientes imágenes de ambas clases para graficar.")
            return

        benign = self.image_df[self.image_df['label'] == 0].sample(n=n, random_state=Config.RANDOM_STATE)
        malignant = self.image_df[self.image_df['label'] == 1].sample(n=n, random_state=Config.RANDOM_STATE)

        fig, axes = plt.subplots(2, n, figsize=(2.2 * n, 4.6))
        fig.suptitle('Ejemplos: BENIGNAS (arriba) vs MALIGNAS (abajo)', fontsize=14, fontweight='bold')

        for i, (_, row) in enumerate(benign.iterrows()):
            img = cv2.imread(row['full_path'], cv2.IMREAD_GRAYSCALE)
            ax = axes[0, i] if n > 1 else axes[0]
            if img is not None:
                ax.imshow(img, cmap='gray')
            ax.set_title(row['type'], fontsize=8)
            ax.axis('off')

        for i, (_, row) in enumerate(malignant.iterrows()):
            img = cv2.imread(row['full_path'], cv2.IMREAD_GRAYSCALE)
            ax = axes[1, i] if n > 1 else axes[1]
            if img is not None:
                ax.imshow(img, cmap='gray')
            ax.set_title(row['type'], fontsize=8)
            ax.axis('off')

        plt.tight_layout()
        plt.savefig(self.results_path / 'ejemplos_benignas_vs_malignas.png', dpi=200, bbox_inches='tight')
        plt.show()

    def save_class_samples_to_disk(self, n=Config.N_TEST_IMAGES_TO_SAVE):
        """Copia una muestra de imágenes de cada clase a
        Resultados/muestras/{benignas,malignas}/ para revisión manual."""
        if self.image_df.empty:
            print("⚠️ Ejecuta build_image_index() primero.")
            return

        print(f"\n💾 Guardando muestra de {n} imágenes por clase en {self.samples_path}")

        for label_value, folder_name in [(0, 'benignas'), (1, 'malignas')]:
            subset = self.image_df[self.image_df['label'] == label_value]
            if len(subset) == 0:
                continue
            sample = subset.sample(n=min(n, len(subset)), random_state=Config.RANDOM_STATE)
            dest_dir = self.samples_path / folder_name
            dest_dir.mkdir(exist_ok=True, parents=True)
            for _, row in sample.iterrows():
                src = Path(row['full_path'])
                if src.exists():
                    dest = dest_dir / f"{row['patient_id']}_{src.name}"
                    shutil.copy(src, dest)
            print(f"   ✅ {folder_name}: {len(sample)} imágenes copiadas en {dest_dir}")

    def generate_summary_report(self):
        print("\n📋 GENERANDO REPORTE RESUMEN")
        print("="*50)

        report = []
        report.append("REPORTE DE ANÁLISIS EXPLORATORIO - DETECCIÓN CÁNCER DE MAMA")
        report.append("="*70)
        report.append(f"Fecha: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
        report.append("")
        report.append("1. DATASET CBIS-DDSM (IMÁGENES MÉDICAS)")
        report.append("-"*40)
        report.append(f"   • Calcificaciones: {len(self.calc_all)} casos")
        report.append(f"   • Masas: {len(self.mass_all)} casos")
        report.append(f"   • Total casos: {len(self.calc_all) + len(self.mass_all)}")

        if 'pathology' in self.calc_all.columns:
            report.append(f"   • Calcificaciones Malignas: {(self.calc_all['pathology']=='MALIGNANT').sum()}")
            report.append(f"   • Calcificaciones Benignas: {(self.calc_all['pathology']=='BENIGN').sum()}")
        if 'pathology' in self.mass_all.columns:
            report.append(f"   • Masas Malignas: {(self.mass_all['pathology']=='MALIGNANT').sum()}")
            report.append(f"   • Masas Benignas: {(self.mass_all['pathology']=='BENIGN').sum()}")

        report.append("")
        report.append("2. DATASET WISCONSIN (DATOS TABULARES)")
        report.append("-"*40)
        report.append(f"   • Total casos: {len(self.wisconsin_data)}")
        if 'diagnosis' in self.wisconsin_data.columns:
            report.append(f"   • Malignos: {(self.wisconsin_data['diagnosis']=='M').sum()}")
            report.append(f"   • Benignos: {(self.wisconsin_data['diagnosis']=='B').sum()}")

        report.append("")
        report.append("3. ÍNDICE DE IMÁGENES CONSTRUIDO")
        report.append("-"*40)
        report.append(f"   • Total imágenes indexadas con ruta real: {len(self.image_df)}")

        report.append("")
        report.append("4. NOTA IMPORTANTE SOBRE LOS DATASETS")
        report.append("-"*40)
        report.append("   CBIS-DDSM (imágenes) y Wisconsin (tabular) son datasets")
        report.append("   INDEPENDIENTES: no comparten pacientes. Por eso los modelos")
        report.append("   'hibridos' de este proyecto combinan dos algoritmos sobre UN")
        report.append("   mismo dataset (imagenes), y no fusionan ambos datasets por caso.")

        report_text = "\n".join(report)
        with open(self.results_path / "exploratory_report.txt", "w") as f:
            f.write(report_text)

        print(report_text)
        print(f"\n💾 Reporte guardado en: {self.results_path / 'exploratory_report.txt'}")

    def run_complete_analysis(self):
        print("🚀 INICIANDO ANÁLISIS EXPLORATORIO COMPLETO (FASE 1)")
        print("="*60)
        try:
            self.load_all_data()
            self.show_data_samples()
            self.analyze_pathology_distribution()
            self.create_distribution_plots()
            num_images = self.check_image_files()
            matching_series = self.map_images_to_metadata()
            self.analyze_clinical_features()

            self.build_image_index()
            if len(self.image_df) > 0:
                self.show_sample_images_by_class()
                self.save_class_samples_to_disk()

            self.generate_summary_report()

            print("\n✅ FASE 1 (EDA) COMPLETADA!")
            print(f"🖼️ Imágenes disponibles: {num_images}")
            print(f"🔗 Series con correspondencia: {matching_series}")
            print(f"📊 Casos indexados con ruta real: {len(self.image_df)}")
            print(f"💾 Resultados guardados en: {self.results_path}")

            return True
        except Exception as e:
            print(f"\n❌ Error durante el análisis: {str(e)}")
            import traceback
            traceback.print_exc()
            return False

In [ ]:
drive_ok = verify_drive_structure()

if drive_ok:
    print("\n🎯 INICIANDO ANÁLISIS EXPLORATORIO...")
    eda = BreastCancerEDA()
    eda_ok = eda.run_complete_analysis()
else:
    print("\n❌ ERROR: No se encontró la carpeta DataSet en Drive")
    print("📍 Asegúrate de que la ruta sea: /content/drive/MyDrive/DataSet")
    eda_ok = False

print("\n" + "="*60)
print("🏁 FASE 1 (EDA) FINALIZADA")
print("="*60)

## 🏗️ FASE 2: Modelado — 3 modelos únicos + 2 modelos híbridos

**Aclaración importante:** CBIS-DDSM (imágenes) y Wisconsin (datos tabulares) son
datasets **independientes** — no comparten pacientes ni un identificador común, por lo
que no es posible fusionar ambos por caso individual. Por eso, en este proyecto, los
**modelos "híbridos" combinan dos algoritmos sobre el mismo dataset de imágenes**
(features profundas extraídas de la CNN + un clasificador clásico), en vez de mezclar
los dos datasets. Si el requisito original pedía específicamente fusionar ambos
datasets por paciente, conviene confirmarlo con quien asignó el trabajo, porque no es
técnicamente posible sin un identificador de paciente compartido.

**Modelos que se entrenan en esta fase:**

| # | Nombre | Tipo | Dataset |
|---|---|---|---|
| 1 | `cnn_efficientnet` | Único | CBIS-DDSM (imágenes) |
| 2 | `tabular_xgboost` | Único | Wisconsin (tabular) |
| 3 | `tabular_rf` | Único | Wisconsin (tabular) |
| 4 | `hybrid_cnn_rf` | Híbrido | CBIS-DDSM (features CNN + Random Forest) |
| 5 | `hybrid_cnn_xgb` | Híbrido | CBIS-DDSM (features CNN + XGBoost) |

**Split de datos:**
- Imágenes: 72% entrenamiento / 8% validación / 20% prueba → agrupado:
  **80% entrenamiento+validación / 20% prueba**.
- Tabular (Wisconsin): **80% entrenamiento / 20% prueba** directo.

**Épocas de la CNN:** `EPOCHS = 12` (bajado desde 20 para acelerar las primeras
corridas de prueba). Si el entrenamiento sigue siendo muy lento en tu Colab, puedes
bajarlo temporalmente a 6-8 solo para validar que el pipeline corre sin errores.

In [ ]:
class BreastCancerDetectionSystem:
    """
    Sistema de modelado: entrena 5 modelos sobre los datos ya explorados en
    la Fase 1 (EDA).

      MODELOS ÚNICOS (3):
        1. cnn_efficientnet  -> CNN (EfficientNetB4) sobre imágenes CBIS-DDSM
        2. tabular_xgboost   -> XGBoost sobre datos tabulares Wisconsin (con GridSearchCV)
        3. tabular_rf        -> Random Forest sobre datos tabulares Wisconsin (con RandomizedSearchCV)

      MODELOS HÍBRIDOS (2) — combinan DOS algoritmos sobre EL MISMO dataset
      de imágenes (features profundas de la CNN + clasificador clásico), ya
      que CBIS-DDSM y Wisconsin no comparten pacientes y no pueden fusionarse
      por caso:
        4. hybrid_cnn_rf     -> features de CNN (1.792 dims, GAP) + Random Forest
        5. hybrid_cnn_xgb    -> features de CNN (1.792 dims, GAP) + XGBoost

    CORRECCIONES aplicadas sobre la versión anterior:
      • Extractor de features usa la capa GlobalAveragePooling2D (1.792
        features) en lugar de la capa Dense de 128 previa a la salida.
      • Extracción de features vectorizada por lotes (tf.data.Dataset +
        predict en batch) en vez de un bucle imagen por imagen.
      • Los híbridos usan el 100% de train_df (sin submuestreo forzado).
      • Tuning de hiperparámetros (GridSearchCV / RandomizedSearchCV) antes
        del entrenamiento final de los modelos tabulares.
      • Métricas completas (accuracy, precision, recall, f1, AUC, matriz de
        confusión) para los 5 modelos.
    """

    def __init__(self, eda):
        self.config = Config()
        self.models = {}
        self.history = {}
        self.results = {}
        self.best_params = {}
        self.cv_results = {}          # se llena en run_cross_validation()
        self.predictions_test = {}    # y_true / y_pred / y_proba por modelo (set de prueba fijo)
        self.eda = eda  # Reutiliza la instancia YA cargada en la Fase 1 (evita recargar datos)

        if self.eda.image_df.empty:
            print("⚠️ El EDA no construyó el índice de imágenes todavía. Ejecutando build_image_index()...")
            self.eda.build_image_index()

        self.image_df = self.eda.image_df

    # ------------------------------------------------------------------
    # Preparación de datos de imágenes: 80% (train+val) / 20% test, y
    # dentro del 80%, 90% train / 10% val -> Train 72% / Val 8% / Test 20%
    # ------------------------------------------------------------------
    def create_data_generators(self):
        print("\n🔄 Creando generadores de datos para imágenes reales...")

        if len(self.image_df) == 0:
            raise ValueError("❌ No hay imágenes para entrenar. Verifica el mapeo de rutas en la Fase 1.")

        train_val_df, test_df = train_test_split(
            self.image_df,
            test_size=self.config.TEST_PCT,
            stratify=self.image_df['label'],
            random_state=self.config.RANDOM_STATE
        )

        val_relative = self.config.VAL_PCT / (self.config.TRAIN_PCT + self.config.VAL_PCT)
        train_df, val_df = train_test_split(
            train_val_df,
            test_size=val_relative,
            stratify=train_val_df['label'],
            random_state=self.config.RANDOM_STATE
        )

        total = len(self.image_df)
        print("📊 División de datos (imágenes):")
        print(f"   • Entrenamiento: {len(train_df)} casos ({len(train_df)/total*100:.1f}%)")
        print(f"   • Validación:    {len(val_df)} casos ({len(val_df)/total*100:.1f}%)")
        print(f"   • Prueba:        {len(test_df)} casos ({len(test_df)/total*100:.1f}%)")
        print(f"   👉 Resumen: {(len(train_df)+len(val_df))/total*100:.0f}% entrenamiento+validación / "
              f"{len(test_df)/total*100:.0f}% prueba")

        self.train_df = train_df.reset_index(drop=True)
        self.val_df = val_df.reset_index(drop=True)
        self.test_df = test_df.reset_index(drop=True)

        self._save_test_split()

        train_transforms = A.Compose([
            A.Resize(self.config.IMG_SIZE, self.config.IMG_SIZE),
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(p=0.3),
            A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.3),
            A.CLAHE(p=0.3),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        val_transforms = A.Compose([
            A.Resize(self.config.IMG_SIZE, self.config.IMG_SIZE),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        self.train_transforms = train_transforms
        self.val_transforms = val_transforms

        def image_generator(df, batch_size, transforms, shuffle=True):
            while True:
                if shuffle:
                    df = df.sample(frac=1).reset_index(drop=True)
                for start in range(0, len(df), batch_size):
                    end = min(start + batch_size, len(df))
                    batch_df = df.iloc[start:end]
                    batch_images, batch_labels = [], []
                    for _, row in batch_df.iterrows():
                        img = self.load_and_preprocess_image(row['full_path'], transforms)
                        if img is not None:
                            batch_images.append(img)
                        else:
                            batch_images.append(np.zeros((self.config.IMG_SIZE, self.config.IMG_SIZE, 3), dtype=np.float32))
                        batch_labels.append(row['label'])
                    yield np.array(batch_images, dtype=np.float32), np.array(batch_labels, dtype=np.float32)

        self.train_generator = image_generator(self.train_df, self.config.BATCH_SIZE, self.train_transforms, shuffle=True)
        self.val_generator = image_generator(self.val_df, self.config.BATCH_SIZE, self.val_transforms, shuffle=False)
        self.test_generator = image_generator(self.test_df, self.config.BATCH_SIZE, self.val_transforms, shuffle=False)

        self.steps_per_epoch = max(1, len(self.train_df) // self.config.BATCH_SIZE)
        self.validation_steps = max(1, len(self.val_df) // self.config.BATCH_SIZE)
        self.test_steps = max(1, len(self.test_df) // self.config.BATCH_SIZE)

        print(f"✅ Generadores listos. Steps/epoch: {self.steps_per_epoch}, Validation steps: {self.validation_steps}")

    def _save_test_split(self):
        """Guarda el split de prueba: metadatos en CSV + muestra física de imágenes."""
        csv_path = self.eda.results_path / "test_set_imagenes.csv"
        self.test_df.to_csv(csv_path, index=False)
        print(f"💾 Metadatos del set de prueba guardados en: {csv_path}")

        dest_root = self.eda.samples_path / "test"
        for label_value, folder_name in [(0, 'benignas'), (1, 'malignas')]:
            subset = self.test_df[self.test_df['label'] == label_value]
            if len(subset) == 0:
                continue
            sample = subset.sample(n=min(self.config.N_TEST_IMAGES_TO_SAVE, len(subset)), random_state=self.config.RANDOM_STATE)
            dest_dir = dest_root / folder_name
            dest_dir.mkdir(exist_ok=True, parents=True)
            for _, row in sample.iterrows():
                src = Path(row['full_path'])
                if src.exists():
                    shutil.copy(src, dest_dir / f"{row['patient_id']}_{src.name}")
            print(f"   💾 Muestra de prueba ({folder_name}): {len(sample)} imágenes copiadas en {dest_dir}")

    def load_and_preprocess_image(self, image_path, transforms=None):
        try:
            if image_path is None or not os.path.exists(image_path):
                return None
            image = cv2.imread(image_path)
            if image is None:
                return None
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            if transforms:
                image = transforms(image=image)['image']
            else:
                image = cv2.resize(image, (self.config.IMG_SIZE, self.config.IMG_SIZE))
                image = image.astype(np.float32) / 255.0
            return image
        except Exception:
            return None

    # ------------------------------------------------------------------
    # MODELO ÚNICO 1: CNN (EfficientNetB4)
    # ------------------------------------------------------------------
    def create_cnn_model(self, backbone='efficientnet'):
        print(f"\n🏗️ Creando modelo CNN ({backbone})...")
        if backbone == 'efficientnet':
            base_model = EfficientNetB4(weights='imagenet', include_top=False,
                                         input_shape=(self.config.IMG_SIZE, self.config.IMG_SIZE, 3))
        elif backbone == 'resnet':
            base_model = ResNet50(weights='imagenet', include_top=False,
                                   input_shape=(self.config.IMG_SIZE, self.config.IMG_SIZE, 3))
        elif backbone == 'densenet':
            base_model = DenseNet121(weights='imagenet', include_top=False,
                                      input_shape=(self.config.IMG_SIZE, self.config.IMG_SIZE, 3))
        else:
            raise ValueError(f"Backbone {backbone} no soportado")

        base_model.trainable = True
        for layer in base_model.layers[:-20]:
            layer.trainable = False

        inputs = keras.Input(shape=(self.config.IMG_SIZE, self.config.IMG_SIZE, 3))
        x = base_model(inputs, training=False)
        x = layers.GlobalAveragePooling2D(name='global_average_pooling2d')(x)
        x = layers.Dropout(0.3)(x)
        x = layers.Dense(256, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.3)(x)
        x = layers.Dense(128, activation='relu')(x)
        x = layers.Dropout(0.2)(x)
        outputs = layers.Dense(1, activation='sigmoid', dtype='float32')(x)

        model = keras.Model(inputs, outputs, name=f'breast_cancer_cnn_{backbone}')
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=self.config.LEARNING_RATE),
            loss='binary_crossentropy',
            metrics=['accuracy', 'precision', 'recall']
        )
        print(f"✅ Modelo CNN creado: {model.count_params():,} parámetros")
        return model

    def train_cnn_model(self, backbone='efficientnet'):
        print(f"\n🚀 ENTRENANDO MODELO CNN ({backbone}) — Modelo único 1/3")
        print("="*50)
        if not hasattr(self, 'train_generator'):
            raise ValueError("❌ No hay generadores de datos. Ejecuta create_data_generators() primero.")

        model = self.create_cnn_model(backbone)
        callbacks = [
            keras.callbacks.EarlyStopping(patience=self.config.PATIENCE, restore_best_weights=True, monitor='val_loss'),
            keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, monitor='val_loss'),
            keras.callbacks.ModelCheckpoint(f"{self.config.MODELS_PATH}/best_cnn_{backbone}.keras",
                                             save_best_only=True, monitor='val_loss')
        ]

        print(f"🚀 Entrenando por {self.config.EPOCHS} épocas, {self.steps_per_epoch} steps/epoch...")
        history = model.fit(
            self.train_generator,
            steps_per_epoch=self.steps_per_epoch,
            epochs=self.config.EPOCHS,
            validation_data=self.val_generator,
            validation_steps=self.validation_steps,
            callbacks=callbacks,
            verbose=1
        )

        self.models['cnn_efficientnet'] = model
        self.history['cnn_efficientnet'] = history.history
        print(f"✅ Entrenamiento completado para {backbone}")
        return model, history

    def load_or_train_cnn_model(self, backbone='efficientnet', force_retrain=False):
        """
        Busca un modelo CNN ya guardado en Config.MODELS_PATH y lo carga en vez
        de reentrenar (que es la parte más lenta del pipeline). Si no encuentra
        ningún archivo (o force_retrain=True), entrena desde cero con
        train_cnn_model().

        Busca, en este orden de preferencia (el más reciente por fecha de
        modificación dentro de cada patrón):
          1. best_cnn_{backbone}.keras      -> checkpoint guardado durante el fit()
          2. cnn_efficientnet_*.keras       -> guardado final de save_models()
          3. best_model_*.keras             -> si la CNN quedó como "mejor modelo global"
        """
        if force_retrain:
            print("🔁 force_retrain=True: se ignoran los archivos existentes y se reentrena.")
            return self.train_cnn_model(backbone)

        search_patterns = [
            f"best_cnn_{backbone}.keras",
            "cnn_efficientnet_*.keras",
            "best_model_*.keras",
        ]
        candidates = []
        for pattern in search_patterns:
            candidates.extend(glob.glob(str(Path(self.config.MODELS_PATH) / pattern)))
        candidates = sorted(set(candidates), key=os.path.getmtime, reverse=True)

        if not candidates:
            print(f"📂 No se encontró ningún modelo CNN guardado en {self.config.MODELS_PATH}. Entrenando desde cero...")
            return self.train_cnn_model(backbone)

        model_path = candidates[0]
        print(f"📂 Modelo CNN ya entrenado encontrado en disco: {model_path}")
        print("   ⏭️  Cargando en vez de reentrenar (usa force_retrain=True para forzar el reentrenamiento).")
        try:
            model = keras.models.load_model(model_path)
            self.models['cnn_efficientnet'] = model
            print(f"✅ CNN cargada desde disco: {model_path}")
            return model, None
        except Exception as e:
            print(f"⚠️ No se pudo cargar el modelo desde disco ({e}). Se entrenará desde cero.")
            return self.train_cnn_model(backbone)

    # ------------------------------------------------------------------
    # MODELO ÚNICO 2: XGBoost tabular (Wisconsin) — split 80/20 + GridSearchCV
    # ------------------------------------------------------------------
    def _get_wisconsin_xy(self):
        wisconsin_clean = self.eda.wisconsin_data.drop(['id', 'Unnamed: 32'], axis=1, errors='ignore').dropna()
        X = wisconsin_clean.drop('diagnosis', axis=1)
        y = (wisconsin_clean['diagnosis'] == 'M').astype(int)
        return X, y

    def create_tabular_model(self, tune=True):
        print("\n📊 Creando modelo tabular XGBoost — Modelo único 2/3")
        print("="*50)

        X, y = self._get_wisconsin_xy()
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=self.config.TABULAR_TEST_SIZE, stratify=y, random_state=self.config.RANDOM_STATE
        )
        print(f"   📊 División Wisconsin: {len(X_train)} entrenamiento ({(1-self.config.TABULAR_TEST_SIZE)*100:.0f}%) / "
              f"{len(X_test)} prueba ({self.config.TABULAR_TEST_SIZE*100:.0f}%)")

        if tune:
            print(f"   🔍 GridSearchCV (XGBoost) sobre {self.config.XGB_PARAM_GRID} con {self.config.TUNING_CV}-fold CV interno...")
            base_xgb = xgb.XGBClassifier(random_state=self.config.RANDOM_STATE, eval_metric='logloss')
            grid = GridSearchCV(
                base_xgb, param_grid=self.config.XGB_PARAM_GRID,
                cv=self.config.TUNING_CV, scoring='roc_auc', n_jobs=-1
            )
            grid.fit(X_train, y_train)
            best_params = grid.best_params_
            print(f"   ✅ Mejores hiperparámetros XGBoost: {best_params} (AUC CV: {grid.best_score_:.3f})")
            self.best_params['tabular_xgboost'] = best_params
            xgb_model = xgb.XGBClassifier(
                **best_params, random_state=self.config.RANDOM_STATE, eval_metric='logloss'
            )
        else:
            xgb_model = xgb.XGBClassifier(
                n_estimators=100, max_depth=6, learning_rate=0.1,
                random_state=self.config.RANDOM_STATE, eval_metric='logloss'
            )

        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

        print("✅ Modelo tabular XGBoost entrenado:")
        print(f"   • Precisión entrenamiento: {xgb_model.score(X_train, y_train):.3f}")
        print(f"   • Precisión test: {xgb_model.score(X_test, y_test):.3f}")

        self.models['tabular_xgboost'] = xgb_model
        self.tabular_data = {'X_train': X_train, 'X_test': X_test, 'y_train': y_train, 'y_test': y_test,
                              'X': X, 'y': y}

        test_export = X_test.copy()
        test_export['diagnosis_real'] = y_test.values
        test_csv = self.eda.results_path / "wisconsin_test_split.csv"
        test_export.to_csv(test_csv, index=False)
        print(f"💾 Muestra de prueba (Wisconsin) guardada en: {test_csv}")

        return xgb_model

    # ------------------------------------------------------------------
    # MODELO ÚNICO 3: Random Forest tabular (Wisconsin) + RandomizedSearchCV
    # ------------------------------------------------------------------
    def train_rf_tabular(self, tune=True):
        print("\n🌲 Entrenando Random Forest tabular — Modelo único 3/3")
        print("="*50)

        if not hasattr(self, 'tabular_data'):
            print("⚠️ Datos tabulares no preparados aún, ejecutando create_tabular_model() primero...")
            self.create_tabular_model()

        X_train, X_test = self.tabular_data['X_train'], self.tabular_data['X_test']
        y_train, y_test = self.tabular_data['y_train'], self.tabular_data['y_test']

        if tune:
            print(f"   🔍 RandomizedSearchCV (Random Forest) sobre {self.config.RF_PARAM_DIST} "
                  f"({self.config.N_ITER_RANDOM} iteraciones, {self.config.TUNING_CV}-fold CV interno)...")
            base_rf = RandomForestClassifier(random_state=self.config.RANDOM_STATE)
            search = RandomizedSearchCV(
                base_rf, param_distributions=self.config.RF_PARAM_DIST,
                n_iter=self.config.N_ITER_RANDOM, cv=self.config.TUNING_CV,
                scoring='roc_auc', random_state=self.config.RANDOM_STATE, n_jobs=-1
            )
            search.fit(X_train, y_train)
            best_params = search.best_params_
            print(f"   ✅ Mejores hiperparámetros Random Forest: {best_params} (AUC CV: {search.best_score_:.3f})")
            self.best_params['tabular_rf'] = best_params
            rf_model = RandomForestClassifier(**best_params, random_state=self.config.RANDOM_STATE)
        else:
            rf_model = RandomForestClassifier(n_estimators=100, random_state=self.config.RANDOM_STATE)

        rf_model.fit(X_train, y_train)

        test_acc = rf_model.score(X_test, y_test)
        print(f"✅ Random Forest tabular entrenado. Precisión test: {test_acc:.3f}")

        self.models['tabular_rf'] = rf_model
        return rf_model

    # ------------------------------------------------------------------
    # Extracción de features profundas (compartida por ambos híbridos)
    # CORRECCIÓN #2: extracción vectorizada por lotes (tf.data + predict batch)
    # en vez de un bucle imagen por imagen (antes tardaba ~30-40 min).
    # ------------------------------------------------------------------
    def extract_features_batched(self, df, feature_extractor, label="", batch_size=None):
        batch_size = batch_size or self.config.FEATURE_BATCH_SIZE
        n = len(df)
        print(f"🧪 Extrayendo características por lotes ({label}) — {n} imágenes, batch_size={batch_size}...")
        if n == 0:
            return np.array([]), np.array([])

        paths = df['full_path'].tolist()
        labels = df['label'].astype(np.float32).tolist()

        def gen():
            for p, lab in zip(paths, labels):
                img = self.load_and_preprocess_image(p, transforms=self.val_transforms)
                if img is None:
                    img = np.zeros((self.config.IMG_SIZE, self.config.IMG_SIZE, 3), dtype=np.float32)
                yield img, lab

        dataset = tf.data.Dataset.from_generator(
            gen,
            output_signature=(
                tf.TensorSpec(shape=(self.config.IMG_SIZE, self.config.IMG_SIZE, 3), dtype=tf.float32),
                tf.TensorSpec(shape=(), dtype=tf.float32)
            )
        ).batch(batch_size).prefetch(tf.data.AUTOTUNE)

        steps = math.ceil(n / batch_size)
        start = time.time()
        features_list, labels_list = [], []
        for imgs, labs in tqdm(dataset, total=steps, desc=f"Extrayendo ({label})"):
            feats = feature_extractor.predict(imgs, verbose=0)
            features_list.append(feats)
            labels_list.append(labs.numpy())

        X = np.concatenate(features_list, axis=0)
        y = np.concatenate(labels_list, axis=0)
        elapsed = time.time() - start
        print(f"✅ {len(X)} imágenes procesadas en {elapsed:.1f}s ({elapsed/max(1,len(X)):.3f}s/img) "
              f"-> {X.shape[1]} features/imagen")
        return X, y

    def _get_feature_extractor(self, backbone='efficientnet'):
        if 'cnn_efficientnet' not in self.models:
            print("⚠️ La CNN base no existe. Entrenándola primero...")
            self.train_cnn_model(backbone)
        cnn_model = self.models['cnn_efficientnet']

        # CORRECCIÓN #1: usar la capa GlobalAveragePooling2D (1.792 features en
        # EfficientNetB4) en lugar de la capa Dropout/Dense de 128 previa a la
        # salida (antes: cnn_model.layers[-2], solo 128 features).
        gap_layer = None
        for layer in cnn_model.layers:
            if isinstance(layer, layers.GlobalAveragePooling2D):
                gap_layer = layer  # se queda con la última (por si hubiera varias)
        if gap_layer is None:
            raise ValueError("❌ No se encontró una capa GlobalAveragePooling2D en el modelo CNN.")

        # NOTA (fix): `layer.output_shape` puede fallar con AttributeError en
        # capas reutilizadas dentro de un modelo funcional (varios "inbound
        # nodes"), típico de TF 2.15+. `layer.output.shape` es la forma
        # robusta de obtener la forma de salida en cualquier versión.
        gap_output = gap_layer.output
        n_features = gap_output.shape[-1]
        print(f"✅ Extractor de features -> capa '{gap_layer.name}' ({n_features} features, antes: 128)")
        return keras.Model(inputs=cnn_model.input, outputs=gap_output)

    def _get_or_extract_hybrid_features(self, feature_extractor):
        """Extrae (o reutiliza si ya se hizo) las features CNN para train y test
        completos. Se cachean en self para reutilizarlas en la CV posterior sin
        tener que re-extraer (evita repetir el paso costoso 5 veces)."""
        if not hasattr(self, 'hybrid_features'):
            self.hybrid_features = {}

        if 'train' not in self.hybrid_features:
            X_train_feat, y_train_feat = self.extract_features_batched(
                self.train_df, feature_extractor, "entrenamiento (100% train_df)"
            )
            self.hybrid_features['train'] = (X_train_feat, y_train_feat)

        if 'test' not in self.hybrid_features:
            X_test_feat, y_test_feat = self.extract_features_batched(
                self.test_df, feature_extractor, "prueba"
            )
            self.hybrid_features['test'] = (X_test_feat, y_test_feat)

        return self.hybrid_features['train'], self.hybrid_features['test']

    # ------------------------------------------------------------------
    # MODELO HÍBRIDO 1: features CNN + Random Forest
    # CORRECCIÓN #3: se elimina el submuestreo forzado (sample_size=2000);
    # ahora se usa el 100% de train_df.
    # ------------------------------------------------------------------
    def train_hybrid_cnn_rf(self, backbone='efficientnet'):
        print(f"\n🔗 Entrenando Híbrido 1/2: CNN ({backbone}) + Random Forest")
        print("="*50)

        feature_extractor = self._get_feature_extractor(backbone)
        (X_feat, y_feat), (X_test_feat, y_test_feat) = self._get_or_extract_hybrid_features(feature_extractor)
        if len(X_feat) == 0:
            raise ValueError("❌ No se pudieron extraer características de ninguna imagen.")

        hybrid_rf = RandomForestClassifier(n_estimators=100, random_state=self.config.RANDOM_STATE)
        hybrid_rf.fit(X_feat, y_feat)

        if len(X_test_feat) > 0:
            test_acc = hybrid_rf.score(X_test_feat, y_test_feat)
            self.results['hybrid_cnn_rf'] = {'test_accuracy': test_acc}
            print(f"   • Precisión en test: {test_acc:.3f}")

        self.models['hybrid_cnn_rf'] = {'extractor': feature_extractor, 'classifier': hybrid_rf}
        print(f"✅ Híbrido CNN+RF entrenado con {len(X_feat)} muestras (100% de train_df).")
        return hybrid_rf

    # ------------------------------------------------------------------
    # MODELO HÍBRIDO 2: features CNN + XGBoost (distinto del híbrido 1)
    # ------------------------------------------------------------------
    def train_hybrid_cnn_xgb(self, backbone='efficientnet'):
        print(f"\n🔗 Entrenando Híbrido 2/2: CNN ({backbone}) + XGBoost")
        print("="*50)

        feature_extractor = self._get_feature_extractor(backbone)
        (X_feat, y_feat), (X_test_feat, y_test_feat) = self._get_or_extract_hybrid_features(feature_extractor)
        if len(X_feat) == 0:
            raise ValueError("❌ No se pudieron extraer características de ninguna imagen.")

        hybrid_xgb = xgb.XGBClassifier(
            n_estimators=150, max_depth=5, learning_rate=0.1,
            random_state=self.config.RANDOM_STATE, eval_metric='logloss'
        )
        hybrid_xgb.fit(X_feat, y_feat)

        if len(X_test_feat) > 0:
            test_acc = hybrid_xgb.score(X_test_feat, y_test_feat)
            self.results['hybrid_cnn_xgb'] = {'test_accuracy': test_acc}
            print(f"   • Precisión en test: {test_acc:.3f}")

        self.models['hybrid_cnn_xgb'] = {'extractor': feature_extractor, 'classifier': hybrid_xgb}
        print(f"✅ Híbrido CNN+XGBoost entrenado con {len(X_feat)} muestras (100% de train_df).")
        return hybrid_xgb

    # ------------------------------------------------------------------
    # Evaluación — CORRECCIÓN #7: precision/recall/f1/matriz de confusión
    # para los 5 modelos, y se guardan las predicciones del set de prueba
    # fijo (self.predictions_test) para las pruebas estadísticas (McNemar/DeLong).
    # ------------------------------------------------------------------
    def evaluate_models(self):
        print("\n📊 EVALUANDO MODELOS (métricas completas)")
        print("="*50)
        results = dict(self.results)  # conserva accuracy de híbridos ya calculada

        def full_metrics(y_true, y_pred, y_proba):
            return {
                'accuracy': accuracy_score(y_true, y_pred),
                'precision': precision_score(y_true, y_pred, zero_division=0),
                'recall': recall_score(y_true, y_pred, zero_division=0),
                'f1': f1_score(y_true, y_pred, zero_division=0),
                'auc': roc_auc_score(y_true, y_proba),
                'confusion_matrix': confusion_matrix(y_true, y_pred).tolist()
            }

        # --- Tabular XGBoost ---
        if 'tabular_xgboost' in self.models:
            model = self.models['tabular_xgboost']
            X_test, y_test = self.tabular_data['X_test'], self.tabular_data['y_test']
            y_pred = model.predict(X_test)
            y_proba = model.predict_proba(X_test)[:, 1]
            results['tabular_xgboost'] = full_metrics(y_test, y_pred, y_proba)
            self.predictions_test['tabular_xgboost'] = {'y_true': np.array(y_test), 'y_pred': y_pred, 'y_proba': y_proba}
            m = results['tabular_xgboost']
            print(f"📊 Tabular XGBoost — Acc: {m['accuracy']:.3f}, Prec: {m['precision']:.3f}, "
                  f"Rec: {m['recall']:.3f}, F1: {m['f1']:.3f}, AUC: {m['auc']:.3f}")

        # --- Tabular Random Forest ---
        if 'tabular_rf' in self.models:
            model = self.models['tabular_rf']
            X_test, y_test = self.tabular_data['X_test'], self.tabular_data['y_test']
            y_pred = model.predict(X_test)
            y_proba = model.predict_proba(X_test)[:, 1]
            results['tabular_rf'] = full_metrics(y_test, y_pred, y_proba)
            self.predictions_test['tabular_rf'] = {'y_true': np.array(y_test), 'y_pred': y_pred, 'y_proba': y_proba}
            m = results['tabular_rf']
            print(f"📊 Tabular Random Forest — Acc: {m['accuracy']:.3f}, Prec: {m['precision']:.3f}, "
                  f"Rec: {m['recall']:.3f}, F1: {m['f1']:.3f}, AUC: {m['auc']:.3f}")

        # --- CNN EfficientNet (evaluada sobre el set de prueba fijo, no solo val) ---
        if 'cnn_efficientnet' in self.models:
            model = self.models['cnn_efficientnet']
            y_true_list, y_proba_list = [], []
            n_batches = math.ceil(len(self.test_df) / self.config.BATCH_SIZE)
            for _ in range(n_batches):
                imgs, labs = next(self.test_generator)
                proba = model.predict(imgs, verbose=0).flatten()
                y_proba_list.append(proba)
                y_true_list.append(labs)
            y_true = np.concatenate(y_true_list)[:len(self.test_df)]
            y_proba = np.concatenate(y_proba_list)[:len(self.test_df)]
            y_pred = (y_proba >= 0.5).astype(int)
            results['cnn_efficientnet'] = full_metrics(y_true, y_pred, y_proba)
            self.predictions_test['cnn_efficientnet'] = {'y_true': y_true, 'y_pred': y_pred, 'y_proba': y_proba}
            if 'cnn_efficientnet' in self.history:
                h = self.history['cnn_efficientnet']
                results['cnn_efficientnet']['train_accuracy'] = h['accuracy'][-1]
                results['cnn_efficientnet']['val_accuracy'] = h['val_accuracy'][-1]
            m = results['cnn_efficientnet']
            print(f"📊 CNN EfficientNet (test) — Acc: {m['accuracy']:.3f}, Prec: {m['precision']:.3f}, "
                  f"Rec: {m['recall']:.3f}, F1: {m['f1']:.3f}, AUC: {m['auc']:.3f}")

        # --- Híbridos ---
        for key in ['hybrid_cnn_rf', 'hybrid_cnn_xgb']:
            if key in self.models:
                classifier = self.models[key]['classifier']
                _, (X_test_feat, y_test_feat) = self._get_or_extract_hybrid_features(self.models[key]['extractor'])
                if len(X_test_feat) > 0:
                    y_pred = classifier.predict(X_test_feat)
                    y_proba = classifier.predict_proba(X_test_feat)[:, 1]
                    results[key] = full_metrics(y_test_feat, y_pred, y_proba)
                    self.predictions_test[key] = {'y_true': y_test_feat, 'y_pred': y_pred, 'y_proba': y_proba}
                    m = results[key]
                    print(f"📊 {key} — Acc: {m['accuracy']:.3f}, Prec: {m['precision']:.3f}, "
                          f"Rec: {m['recall']:.3f}, F1: {m['f1']:.3f}, AUC: {m['auc']:.3f}")

        self.results = results
        return results

    # ------------------------------------------------------------------
    # Guardado: UN archivo por modelo (los híbridos se empaquetan en .zip)
    # CORRECCIÓN F: además se guarda el mejor modelo (.keras + .tflite)
    # ------------------------------------------------------------------
    def save_models(self):
        print("\n💾 GUARDANDO MODELOS (1 archivo por modelo)")
        print("="*50)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        models_saved = 0
        saved_paths = {}

        for name, model in self.models.items():
            try:
                if isinstance(model, dict) and 'extractor' in model and 'classifier' in model:
                    # Empaquetar extractor (.keras) + clasificador (.pkl) en UN solo .zip
                    tmp_dir = Path(self.config.MODELS_PATH) / f"_tmp_{name}"
                    tmp_dir.mkdir(exist_ok=True, parents=True)
                    extractor_path = tmp_dir / "extractor.keras"
                    classifier_path = tmp_dir / "classifier.pkl"
                    model['extractor'].save(extractor_path)
                    joblib.dump(model['classifier'], classifier_path)

                    zip_path = Path(self.config.MODELS_PATH) / f"{name}_{timestamp}.zip"
                    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
                        zf.write(extractor_path, arcname="extractor.keras")
                        zf.write(classifier_path, arcname="classifier.pkl")
                    shutil.rmtree(tmp_dir)

                    print(f"✅ {name} (híbrido) guardado como archivo único: {zip_path}")
                    saved_paths[name] = str(zip_path)
                    models_saved += 1

                elif hasattr(model, 'save'):
                    model_path = Path(self.config.MODELS_PATH) / f"{name}_{timestamp}.keras"
                    model.save(model_path)
                    print(f"✅ {name} guardado en: {model_path}")
                    saved_paths[name] = str(model_path)
                    models_saved += 1

                else:
                    model_path = Path(self.config.MODELS_PATH) / f"{name}_{timestamp}.pkl"
                    joblib.dump(model, model_path)
                    print(f"✅ {name} guardado en: {model_path}")
                    saved_paths[name] = str(model_path)
                    models_saved += 1

            except Exception as e:
                print(f"⚠️ Error guardando {name}: {e}")

        # ---- Mejor modelo global según Accuracy en test ----
        best_name, best_path = self._save_best_model(timestamp)
        if best_name:
            saved_paths['best_model'] = best_path

        final_report = {
            'timestamp': timestamp,
            'models_trained': list(self.models.keys()),
            'saved_paths': saved_paths,
            'model_results': self.results,
            'best_model': best_name,
            'best_params': self.best_params
        }
        results_path = Path(self.config.RESULTS_PATH) / f"system_save_report_{timestamp}.json"
        with open(results_path, 'w') as f:
            json.dump(final_report, f, indent=2, default=str)
        print(f"✅ Reporte de guardado generado en: {results_path}")

        return models_saved, final_report

    def _save_best_model(self, timestamp):
        """Determina el mejor modelo por Accuracy en test y lo guarda en
        formato .keras (nativo) y .tflite (para despliegue), cuando aplica.
        Para los híbridos (RF/XGB), se guarda únicamente el .zip ya generado
        y se referencia en el metadato."""
        if not self.results:
            print("⚠️ No hay resultados para determinar el mejor modelo.")
            return None, None

        acc_by_model = {k: v.get('accuracy', v.get('test_accuracy', 0)) for k, v in self.results.items()}
        if not acc_by_model:
            return None, None

        best_name = max(acc_by_model, key=acc_by_model.get)
        best_acc = acc_by_model[best_name]
        print(f"\n🏆 Mejor modelo global: '{best_name}' (Accuracy test = {best_acc:.3f})")

        best_path = None
        if best_name == 'cnn_efficientnet':
            keras_path = Path(self.config.MODELS_PATH) / f"best_model_{timestamp}.keras"
            self.models['cnn_efficientnet'].save(keras_path)
            print(f"✅ Mejor modelo (.keras) guardado en: {keras_path}")

            try:
                converter = tf.lite.TFLiteConverter.from_keras_model(self.models['cnn_efficientnet'])
                converter.optimizations = [tf.lite.Optimize.DEFAULT]
                tflite_model = converter.convert()
                tflite_path = Path(self.config.MODELS_PATH) / f"best_model_{timestamp}.tflite"
                with open(tflite_path, 'wb') as f:
                    f.write(tflite_model)
                print(f"✅ Mejor modelo (.tflite) guardado en: {tflite_path}")
                best_path = str(tflite_path)
            except Exception as e:
                print(f"⚠️ No se pudo convertir a TFLite: {e}")
                best_path = str(keras_path)
        else:
            print(f"   ℹ️ '{best_name}' no es una CNN — ya quedó guardado en su formato nativo "
                  f"(.pkl o .zip) arriba. Se referencia como mejor modelo en el reporte.")

        return best_name, best_path

    # ------------------------------------------------------------------
    # Pipeline completo
    # ------------------------------------------------------------------
    def run_complete_training(self):
        print("🚀 INICIANDO ENTRENAMIENTO COMPLETO (FASE 2) — 3 modelos únicos + 2 híbridos")
        print("="*60)

        try:
            self.create_data_generators()

            # Modelos únicos (con tuning de hiperparámetros)
            self.create_tabular_model(tune=True)
            self.train_rf_tabular(tune=True)
            self.train_cnn_model('efficientnet')

            # Modelos híbridos (100% de train_df, extracción por lotes)
            self.train_hybrid_cnn_rf('efficientnet')
            self.train_hybrid_cnn_xgb('efficientnet')

            self.evaluate_models()
            models_saved, final_report = self.save_models()

            print("\n🎉 ¡ENTRENAMIENTO COMPLETO EXITOSO!")
            print(f"   • Modelos entrenados: {len(self.models)} (esperado: 5)")
            print(f"   • Modelos guardados: {models_saved}")
            print(f"   • Imágenes usadas: {len(self.image_df)}")

            return True
        except Exception as e:
            print(f"\n❌ Error durante el entrenamiento: {str(e)}")
            import traceback
            traceback.print_exc()
            return False


## 🏗️ FASE 2: Entrenamiento — una celda por modelo

**Cambio importante respecto a la versión anterior:** antes todo el
entrenamiento (datos + 3 modelos únicos + 2 híbridos + evaluación + guardado)
corría dentro de una sola función `run_complete_training()`. Si un modelo
fallaba (como el bug de `output_shape` en los híbridos), había que volver a
ejecutar **todo** desde cero, incluyendo la CNN (la parte más lenta).

Ahora cada modelo se entrena en su **propia celda**, en este orden:

1. Inicializar `system` + generadores de datos.
2. XGBoost tabular (con tuning).
3. Random Forest tabular (con tuning).
4. CNN EfficientNetB4.
5. Híbrido CNN + Random Forest.
6. Híbrido CNN + XGBoost.
7. Evaluación de los 5 modelos.
8. Guardado de modelos + mejor modelo.

Mientras el kernel de Colab no se reinicie, `system` conserva en memoria todo lo
ya entrenado. Si una celda falla (por ejemplo, un híbrido), solo necesitas
corregir el error y **volver a ejecutar esa celda** — no hace falta repetir las
anteriores. Cada celda verifica sus prerequisitos con mensajes claros.


In [ ]:
print("🎯 INICIANDO DESARROLLO DEL SISTEMA DE DETECCIÓN (FASE 2)")
print("="*60)

if 'eda_ok' not in globals() or not eda_ok or eda.image_df.empty:
    raise RuntimeError("❌ La Fase 1 (EDA) no se completó correctamente o no se indexaron imágenes. "
                        "Revisa la sección de verificación de Drive antes de continuar.")

if 'system' not in globals():
    system = BreastCancerDetectionSystem(eda)
    print("✅ 'system' creado.")
else:
    print("ℹ️ 'system' ya existía en memoria, se reutiliza (no se pierde lo ya entrenado).")

if not hasattr(system, 'train_df'):
    system.create_data_generators()
else:
    print("ℹ️ Los generadores de datos ya estaban creados, no se recrean "
          "(evita cambiar el split train/val/test entre reintentos).")


In [ ]:
if 'system' not in globals():
    raise RuntimeError("⚠️ Ejecuta primero la celda de inicialización de 'system'.")

if 'tabular_xgboost' in system.models:
    print("ℹ️ 'tabular_xgboost' ya está entrenado. Si quieres reentrenarlo, borra "
          "system.models['tabular_xgboost'] y vuelve a correr esta celda.")
else:
    try:
        system.create_tabular_model(tune=True)
    except Exception as e:
        print(f"❌ Error entrenando XGBoost tabular: {e}")
        import traceback; traceback.print_exc()


In [ ]:
if 'system' not in globals():
    raise RuntimeError("⚠️ Ejecuta primero la celda de inicialización de 'system'.")

if 'tabular_rf' in system.models:
    print("ℹ️ 'tabular_rf' ya está entrenado. Si quieres reentrenarlo, borra "
          "system.models['tabular_rf'] y vuelve a correr esta celda.")
else:
    try:
        system.train_rf_tabular(tune=True)
    except Exception as e:
        print(f"❌ Error entrenando Random Forest tabular: {e}")
        import traceback; traceback.print_exc()


In [ ]:
if 'system' not in globals():
    raise RuntimeError("⚠️ Ejecuta primero la celda de inicialización de 'system'.")

if 'cnn_efficientnet' in system.models:
    print("ℹ️ 'cnn_efficientnet' ya está en memoria. Si quieres reentrenarla, borra "
          "system.models['cnn_efficientnet'] y vuelve a correr esta celda.")
else:
    try:
        # Busca automáticamente un .keras ya guardado en Config.MODELS_PATH
        # (best_cnn_efficientnet.keras / cnn_efficientnet_*.keras / best_model_*.keras).
        # Si lo encuentra, lo CARGA y se salta el entrenamiento. Si no existe
        # ninguno, entrena desde cero. Para forzar reentrenamiento aunque exista
        # el archivo, usa: system.load_or_train_cnn_model('efficientnet', force_retrain=True)
        system.load_or_train_cnn_model('efficientnet')
    except Exception as e:
        print(f"❌ Error entrenando/cargando la CNN: {e}")
        import traceback; traceback.print_exc()


In [ ]:
if 'system' not in globals():
    raise RuntimeError("⚠️ Ejecuta primero la celda de inicialización de 'system'.")
if 'cnn_efficientnet' not in system.models:
    raise RuntimeError("⚠️ Entrena primero la CNN (celda anterior) — los híbridos reutilizan sus features.")

if 'hybrid_cnn_rf' in system.models:
    print("ℹ️ 'hybrid_cnn_rf' ya está entrenado. Si quieres reentrenarlo, borra "
          "system.models['hybrid_cnn_rf'] y vuelve a correr esta celda.")
else:
    try:
        system.train_hybrid_cnn_rf('efficientnet')
    except Exception as e:
        print(f"❌ Error entrenando Híbrido CNN+RF: {e}")
        import traceback; traceback.print_exc()


In [ ]:
if 'system' not in globals():
    raise RuntimeError("⚠️ Ejecuta primero la celda de inicialización de 'system'.")
if 'cnn_efficientnet' not in system.models:
    raise RuntimeError("⚠️ Entrena primero la CNN — los híbridos reutilizan sus features.")

if 'hybrid_cnn_xgb' in system.models:
    print("ℹ️ 'hybrid_cnn_xgb' ya está entrenado. Si quieres reentrenarlo, borra "
          "system.models['hybrid_cnn_xgb'] y vuelve a correr esta celda.")
else:
    try:
        # Si ya se extrajeron features para el híbrido RF, se reutilizan aquí
        # (ver system.hybrid_features) y este paso es rápido.
        system.train_hybrid_cnn_xgb('efficientnet')
    except Exception as e:
        print(f"❌ Error entrenando Híbrido CNN+XGBoost: {e}")
        import traceback; traceback.print_exc()


In [ ]:
if 'system' not in globals():
    raise RuntimeError("⚠️ Ejecuta primero las celdas de entrenamiento.")

expected = ['tabular_xgboost', 'tabular_rf', 'cnn_efficientnet', 'hybrid_cnn_rf', 'hybrid_cnn_xgb']
faltantes = [m for m in expected if m not in system.models]
if faltantes:
    print(f"⚠️ Aún faltan por entrenar: {faltantes}. Puedes evaluar igual con los modelos "
          f"disponibles ({len(system.models)}/5), o completar el entrenamiento primero.")

try:
    system.evaluate_models()
except Exception as e:
    print(f"❌ Error evaluando modelos: {e}")
    import traceback; traceback.print_exc()


In [ ]:
if 'system' not in globals():
    raise RuntimeError("⚠️ Ejecuta primero las celdas de entrenamiento y evaluación.")
if not system.results:
    raise RuntimeError("⚠️ Ejecuta primero la celda de evaluación (system.evaluate_models()).")

try:
    models_saved, final_report = system.save_models()
    print("\n🏆 ¡SISTEMA DE DETECCIÓN DE CÁNCER COMPLETADO!")
    print(f"   • Modelos entrenados: {len(system.models)} (esperado: 5)")
    print(f"   • Modelos guardados: {models_saved}")
    print(f"   • Imágenes usadas: {len(system.image_df)}")
    print("\n💾 ARCHIVOS GENERADOS:")
    print(f"   📁 Modelos: {system.config.MODELS_PATH}/")
    print(f"   📁 Resultados y muestras: {system.config.RESULTS_PATH}/")
except Exception as e:
    print(f"❌ Error guardando modelos: {e}")
    import traceback; traceback.print_exc()

print("\n" + "="*60)
print("🏁 FASE 2 FINALIZADA")
print("="*60)


## 📊 Validación Cruzada (Cross-Validation) para los 4 modelos comparables

La validación cruzada es una técnica robusta para evaluar el rendimiento de un
modelo, dividiendo el dataset en múltiples subconjuntos (folds). El modelo se
entrena en un subconjunto y se evalúa en el resto, repitiendo este proceso varias
veces con diferentes combinaciones. Esto proporciona una estimación más fiable de
cómo el modelo generalizará a datos no vistos, reduciendo el riesgo de sobreajuste.

**CORRECCIÓN #5:** antes solo se corría CV para XGBoost tabular. Ahora se ejecuta
`StratifiedKFold` de {N_FOLDS} folds para los **4 modelos comparables**:

1. **XGBoost tabular** (Wisconsin) — CV sobre `X`, `y` originales.
2. **Random Forest tabular** (Wisconsin) — CV sobre `X`, `y` originales.
3. **Hybrid CNN+RF** — CV sobre las 1.792 features ya extraídas de `train_df`
   (no se re-entrena la CNN en cada fold, solo el clasificador clásico —
   esto es lo que hace viable la CV para los híbridos sin costo prohibitivo).
4. **Hybrid CNN+XGB** — ídem, sobre las mismas features.

La CNN pura (`cnn_efficientnet`) queda fuera de esta CV porque reentrenarla desde
cero 5 veces (fine-tuning de EfficientNetB4) sí sería demasiado costoso para el
tiempo disponible; se evalúa con el split fijo train/val/test como antes.

Los resultados (accuracy, AUC y F1 por fold) se guardan en `system.cv_results`
para alimentar después las pruebas estadísticas (t-pareada / Wilcoxon).


In [ ]:
if 'system' not in globals():
    raise RuntimeError("⚠️ Ejecuta primero la celda de entrenamiento completo (FASE 2) para crear 'system'.")

def run_cross_validation(system):
    """
    CORRECCIÓN #5: Ejecuta 5-Fold CV para los 4 modelos comparables
    (XGBoost tabular, RF tabular, Hybrid CNN+RF, Hybrid CNN+XGB) y guarda
    accuracy / AUC / F1 de cada fold en system.cv_results.
    """
    cfg = system.config
    skf = StratifiedKFold(n_splits=cfg.N_FOLDS, shuffle=True, random_state=cfg.RANDOM_STATE)
    cv_results = {}

    # ---------- 1) Modelos TABULARES (Wisconsin) ----------
    X, y = system._get_wisconsin_xy()
    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)

    tabular_specs = {
        'tabular_xgboost': lambda: xgb.XGBClassifier(
            **system.best_params.get('tabular_xgboost', dict(n_estimators=100, max_depth=6, learning_rate=0.1)),
            random_state=cfg.RANDOM_STATE, eval_metric='logloss'
        ),
        'tabular_rf': lambda: RandomForestClassifier(
            **system.best_params.get('tabular_rf', dict(n_estimators=100)),
            random_state=cfg.RANDOM_STATE
        ),
    }

    for name, make_model in tabular_specs.items():
        print(f"\n🚀 Validación Cruzada ({cfg.N_FOLDS}-Fold) — {name}")
        accs, aucs, f1s = [], [], []
        for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), start=1):
            X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
            y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]
            model = make_model()
            model.fit(X_tr, y_tr)
            y_pred = model.predict(X_te)
            y_proba = model.predict_proba(X_te)[:, 1]
            acc = accuracy_score(y_te, y_pred)
            auc = roc_auc_score(y_te, y_proba)
            f1 = f1_score(y_te, y_pred, zero_division=0)
            accs.append(acc); aucs.append(auc); f1s.append(f1)
            print(f"   Fold {fold}: Accuracy={acc:.3f}  AUC={auc:.3f}  F1={f1:.3f}")
        cv_results[name] = {'accuracy': accs, 'auc': aucs, 'f1': f1s}
        print(f"   ✅ Media -> Accuracy={np.mean(accs):.3f}±{np.std(accs):.3f}  "
              f"AUC={np.mean(aucs):.3f}±{np.std(aucs):.3f}  F1={np.mean(f1s):.3f}±{np.std(f1s):.3f}")

    # ---------- 2) Modelos HÍBRIDOS (sobre features ya extraídas, sin re-entrenar la CNN) ----------
    hybrid_specs = {
        'hybrid_cnn_rf': lambda: RandomForestClassifier(n_estimators=100, random_state=cfg.RANDOM_STATE),
        'hybrid_cnn_xgb': lambda: xgb.XGBClassifier(
            n_estimators=150, max_depth=5, learning_rate=0.1,
            random_state=cfg.RANDOM_STATE, eval_metric='logloss'
        ),
    }

    if hasattr(system, 'hybrid_features') and 'train' in system.hybrid_features:
        X_feat, y_feat = system.hybrid_features['train']
        for name, make_model in hybrid_specs.items():
            print(f"\n🚀 Validación Cruzada ({cfg.N_FOLDS}-Fold) — {name} (sobre features CNN, 1.792 dims)")
            accs, aucs, f1s = [], [], []
            for fold, (tr_idx, te_idx) in enumerate(skf.split(X_feat, y_feat), start=1):
                X_tr, X_te = X_feat[tr_idx], X_feat[te_idx]
                y_tr, y_te = y_feat[tr_idx], y_feat[te_idx]
                model = make_model()
                model.fit(X_tr, y_tr)
                y_pred = model.predict(X_te)
                y_proba = model.predict_proba(X_te)[:, 1]
                acc = accuracy_score(y_te, y_pred)
                auc = roc_auc_score(y_te, y_proba)
                f1 = f1_score(y_te, y_pred, zero_division=0)
                accs.append(acc); aucs.append(auc); f1s.append(f1)
                print(f"   Fold {fold}: Accuracy={acc:.3f}  AUC={auc:.3f}  F1={f1:.3f}")
            cv_results[name] = {'accuracy': accs, 'auc': aucs, 'f1': f1s}
            print(f"   ✅ Media -> Accuracy={np.mean(accs):.3f}±{np.std(accs):.3f}  "
                  f"AUC={np.mean(aucs):.3f}±{np.std(aucs):.3f}  F1={np.mean(f1s):.3f}±{np.std(f1s):.3f}")
    else:
        print("\n⚠️ No se encontraron features de híbridos cacheadas en 'system.hybrid_features'. "
              "Ejecuta primero train_hybrid_cnn_rf/xgb (FASE 2) para poder correr su CV.")

    system.cv_results = cv_results
    return cv_results


cv_results = run_cross_validation(system)

print("\n" + "="*60)
print("📋 RESUMEN DE VALIDACIÓN CRUZADA (5-Fold) — 4 modelos")
print("="*60)
summary_rows = []
for name, metrics in cv_results.items():
    summary_rows.append({
        'Modelo': name,
        'Accuracy (media)': np.mean(metrics['accuracy']),
        'Accuracy (std)': np.std(metrics['accuracy']),
        'AUC (media)': np.mean(metrics['auc']),
        'AUC (std)': np.std(metrics['auc']),
        'F1 (media)': np.mean(metrics['f1']),
        'F1 (std)': np.std(metrics['f1']),
    })
cv_summary_df = pd.DataFrame(summary_rows)
display(cv_summary_df)


### Visualización de Métricas de Validación Cruzada (4 modelos)

Los siguientes gráficos muestran la Accuracy, AUC y F1-Score por 'fold' de la
validación cruzada para los 4 modelos comparables, permitiendo visualizar tanto
el rendimiento medio como la consistencia (dispersión) de cada uno.


In [ ]:
if 'cv_results' not in globals():
    raise RuntimeError("⚠️ Ejecuta primero la celda de Validación Cruzada.")

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
metric_titles = {'accuracy': 'Accuracy', 'auc': 'AUC-ROC', 'f1': 'F1-Score'}
palette = sns.color_palette('husl', len(cv_results))

for ax, (metric_key, metric_label) in zip(axes, metric_titles.items()):
    plot_rows = []
    for name, metrics in cv_results.items():
        for fold_i, val in enumerate(metrics[metric_key], start=1):
            plot_rows.append({'Modelo': name, 'Fold': f"Fold {fold_i}", metric_label: val})
    plot_df = pd.DataFrame(plot_rows)
    sns.boxplot(data=plot_df, x='Modelo', y=metric_label, ax=ax, palette=palette)
    sns.stripplot(data=plot_df, x='Modelo', y=metric_label, ax=ax, color='black', alpha=0.6, size=5)
    ax.set_title(f'{metric_label} por Fold — 5-Fold CV', fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
    ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.suptitle('Resultados de Validación Cruzada — 4 Modelos Comparables', y=1.04, fontsize=16, fontweight='bold')

cv_fig_path = Path(system.config.RESULTS_PATH) / "cv_comparison_boxplots.png"
plt.savefig(cv_fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"💾 Figura guardada en: {cv_fig_path}")

# Barras con media ± std (accuracy y AUC)
fig2, axes2 = plt.subplots(1, 2, figsize=(16, 6))
names = list(cv_results.keys())
for ax, metric_key, metric_label in zip(axes2, ['accuracy', 'auc'], ['Accuracy', 'AUC']):
    means = [np.mean(cv_results[n][metric_key]) for n in names]
    stds = [np.std(cv_results[n][metric_key]) for n in names]
    ax.bar(names, means, yerr=stds, capsize=6, color=palette)
    ax.set_title(f'{metric_label} media ± std (5-Fold CV)', fontsize=13, fontweight='bold')
    ax.set_ylim(min(means) * 0.9, 1.0)
    ax.tick_params(axis='x', rotation=30)
    ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
cv_bar_path = Path(system.config.RESULTS_PATH) / "cv_comparison_bars.png"
plt.savefig(cv_bar_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"💾 Figura guardada en: {cv_bar_path}")


### Importancia de Características del Modelo Tabular

Uno de los 'parámetros' clave de un modelo de árboles como XGBoost es la
importancia de las características (feature importance). Este gráfico ayuda a
entender qué variables del dataset de Wisconsin fueron más relevantes para las
predicciones del modelo.

In [ ]:
if 'system' not in globals() or 'tabular_xgboost' not in system.models:
    raise RuntimeError("⚠️ Ejecuta primero la celda de entrenamiento completo (FASE 2).")

wisconsin_clean = system.eda.wisconsin_data.drop(['id', 'Unnamed: 32'], axis=1, errors='ignore').dropna()
X = wisconsin_clean.drop('diagnosis', axis=1)

best_xgb_model = system.models['tabular_xgboost']
importance = best_xgb_model.feature_importances_
feature_importance_df = pd.DataFrame({'Característica': X.columns, 'Importancia': importance})
feature_importance_df = feature_importance_df.sort_values(by='Importancia', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(x='Importancia', y='Característica', data=feature_importance_df, palette='cividis')
plt.title('Importancia de Características (Modelo Tabular XGBoost)', fontsize=16)
plt.xlabel('Importancia (Gain)', fontsize=12)
plt.ylabel('Característica', fontsize=12)
plt.tight_layout()
plt.show()

## 🧪 Pruebas Estadísticas (CORRECCIÓN #6)

Con las métricas por fold (`system.cv_results`) y las predicciones sobre el set
de prueba fijo (`system.predictions_test`) ya disponibles, se aplican tres
pruebas estadísticas para comparar los 5 modelos entre sí:

1. **Test de McNemar** — compara Accuracy en el **set de prueba fijo (20%)**
   entre cada par de modelos, usando la tabla de contingencia de aciertos/fallos
   por caso.
2. **Prueba t-pareada (o Wilcoxon si no se cumple normalidad)** — compara
   Accuracy y F1 usando los **5 resultados de cada fold** de la CV. Se aplica
   **corrección de Bonferroni** (α = 0.05 / 10 = 0.005) por las 10 comparaciones
   posibles entre los 4 modelos con CV (XGB tab., RF tab., Hybrid RF, Hybrid XGB).
3. **Test de DeLong** — compara curvas AUC-ROC entre pares de modelos sobre el
   set de prueba fijo (implementación analítica con `scipy.stats.norm`, sin
   necesidad de librerías externas).

Todo esto se guarda en una **matriz de p-valores 5×5** por prueba, que luego
alimenta el reporte en Excel/Word.


In [ ]:
if 'system' not in globals() or not system.predictions_test:
    raise RuntimeError("⚠️ Ejecuta primero evaluate_models() (FASE 2) para tener system.predictions_test.")

# --------------------------------------------------------------------
# McNemar (accuracy, set de prueba fijo) y DeLong (AUC, set de prueba fijo)
# Solo tienen sentido estadístico ENTRE modelos evaluados sobre el MISMO
# conjunto de casos. Por eso se agrupan por dominio:
#   • Dominio tabular (Wisconsin): tabular_xgboost, tabular_rf
#   • Dominio imágenes (CBIS-DDSM): cnn_efficientnet, hybrid_cnn_rf, hybrid_cnn_xgb
# Los pares entre dominios distintos no son comparables caso a caso y se
# dejan como NaN en las matrices.
# --------------------------------------------------------------------
DOMAIN_GROUPS = {
    'tabular': ['tabular_xgboost', 'tabular_rf'],
    'imagen': ['cnn_efficientnet', 'hybrid_cnn_rf', 'hybrid_cnn_xgb'],
}
ALL_MODELS = [m for m in system.predictions_test.keys()]


def mcnemar_test(y_true, pred_a, pred_b):
    """McNemar con corrección de continuidad sobre aciertos/fallos pareados."""
    correct_a = (pred_a == y_true)
    correct_b = (pred_b == y_true)
    n01 = int(np.sum(correct_a & ~correct_b))   # A acierta, B falla
    n10 = int(np.sum(~correct_a & correct_b))   # A falla, B acierta
    if n01 + n10 == 0:
        return 0.0, 1.0
    stat = (abs(n01 - n10) - 1) ** 2 / (n01 + n10)
    p = 1 - chi2.cdf(stat, df=1)
    return stat, p


def _compute_midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T
    return T2


def _fast_delong(predictions_sorted_transposed, label_1_count):
    m = label_1_count
    n = predictions_sorted_transposed.shape[1] - m
    positive_examples = predictions_sorted_transposed[:, :m]
    negative_examples = predictions_sorted_transposed[:, m:]
    k = predictions_sorted_transposed.shape[0]

    tx = np.empty([k, m], dtype=float)
    ty = np.empty([k, n], dtype=float)
    tz = np.empty([k, m + n], dtype=float)
    for r in range(k):
        tx[r, :] = _compute_midrank(positive_examples[r, :])
        ty[r, :] = _compute_midrank(negative_examples[r, :])
        tz[r, :] = _compute_midrank(predictions_sorted_transposed[r, :])
    aucs = tz[:, :m].sum(axis=1) / m / n - float(m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx[:, :]) / n
    v10 = 1.0 - (tz[:, m:] - ty[:, :]) / m
    sx = np.cov(v01)
    sy = np.cov(v10)
    delongcov = sx / m + sy / n
    return aucs, delongcov


def delong_test(y_true, prob_a, prob_b):
    """Test de DeLong: compara dos AUC correlacionadas sobre el mismo set."""
    y_true = np.asarray(y_true)
    order = np.argsort(-y_true)
    y_true_sorted = y_true[order]
    prob_a_sorted = np.asarray(prob_a)[order]
    prob_b_sorted = np.asarray(prob_b)[order]
    label_1_count = int(np.sum(y_true_sorted))
    if label_1_count == 0 or label_1_count == len(y_true_sorted):
        return np.nan, np.nan, np.nan, np.nan
    preds = np.vstack([prob_a_sorted, prob_b_sorted])
    aucs, cov = _fast_delong(preds, label_1_count)
    auc_diff = aucs[0] - aucs[1]
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    if var <= 0:
        return aucs[0], aucs[1], 0.0, 1.0
    z = auc_diff / np.sqrt(var)
    p = 2 * (1 - norm.cdf(abs(z)))
    return aucs[0], aucs[1], z, p


mcnemar_matrix = pd.DataFrame(np.nan, index=ALL_MODELS, columns=ALL_MODELS)
delong_matrix = pd.DataFrame(np.nan, index=ALL_MODELS, columns=ALL_MODELS)

print("🚀 Calculando McNemar y DeLong (pares dentro del mismo dominio de datos)...\n")
for group_name, models_in_group in DOMAIN_GROUPS.items():
    models_in_group = [m for m in models_in_group if m in system.predictions_test]
    for i in range(len(models_in_group)):
        for j in range(i + 1, len(models_in_group)):
            a, b = models_in_group[i], models_in_group[j]
            pa = system.predictions_test[a]
            pb = system.predictions_test[b]
            # McNemar requiere mismas etiquetas verdaderas y mismo orden de casos
            if len(pa['y_true']) == len(pb['y_true']):
                _, p_mcnemar = mcnemar_test(np.asarray(pa['y_true']), pa['y_pred'], pb['y_pred'])
                mcnemar_matrix.loc[a, b] = p_mcnemar
                mcnemar_matrix.loc[b, a] = p_mcnemar

                _, _, _, p_delong = delong_test(pa['y_true'], pa['y_proba'], pb['y_proba'])
                delong_matrix.loc[a, b] = p_delong
                delong_matrix.loc[b, a] = p_delong

                sig_m = "significativo" if p_mcnemar < system.config.ALPHA else "no significativo"
                sig_d = "significativo" if p_delong < system.config.ALPHA else "no significativo"
                print(f"   {a} vs {b}: McNemar p={p_mcnemar:.4f} ({sig_m}) | DeLong p={p_delong:.4f} ({sig_d})")

print("\n📋 Matriz de p-valores — McNemar (Accuracy, set de prueba fijo):")
display(mcnemar_matrix.round(4))
print("\n📋 Matriz de p-valores — DeLong (AUC, set de prueba fijo):")
display(delong_matrix.round(4))


In [ ]:
if 'cv_results' not in globals():
    raise RuntimeError("⚠️ Ejecuta primero la celda de Validación Cruzada (cv_results).")

cv_models = list(cv_results.keys())
ttest_matrix = pd.DataFrame(np.nan, index=cv_models, columns=cv_models)
wilcoxon_matrix = pd.DataFrame(np.nan, index=cv_models, columns=cv_models)

print(f"🚀 Prueba t-pareada / Wilcoxon sobre Accuracy de los {system.config.N_FOLDS} folds de CV")
print(f"   Corrección de Bonferroni: α = {system.config.ALPHA} / {system.config.N_COMPARISONS} "
      f"= {system.config.ALPHA_BONFERRONI:.4f}\n")

pairs_summary = []
for i in range(len(cv_models)):
    for j in range(i + 1, len(cv_models)):
        a, b = cv_models[i], cv_models[j]
        acc_a = np.array(cv_results[a]['accuracy'])
        acc_b = np.array(cv_results[b]['accuracy'])

        t_stat, p_ttest = ttest_rel(acc_a, acc_b)
        ttest_matrix.loc[a, b] = p_ttest
        ttest_matrix.loc[b, a] = p_ttest

        diff = acc_a - acc_b
        if np.all(diff == 0):
            p_wilcoxon = 1.0
        else:
            try:
                _, p_wilcoxon = wilcoxon(acc_a, acc_b)
            except ValueError:
                p_wilcoxon = np.nan
        wilcoxon_matrix.loc[a, b] = p_wilcoxon
        wilcoxon_matrix.loc[b, a] = p_wilcoxon

        sig = "significativo (Bonferroni)" if p_ttest < system.config.ALPHA_BONFERRONI else "no significativo"
        print(f"   {a} vs {b}: t-pareada p={p_ttest:.4f} | Wilcoxon p={p_wilcoxon:.4f}  -> {sig}")
        pairs_summary.append({
            'Modelo A': a, 'Modelo B': b,
            'Accuracy media A': acc_a.mean(), 'Accuracy media B': acc_b.mean(),
            'p-valor (t-pareada)': p_ttest, 'p-valor (Wilcoxon)': p_wilcoxon,
            'Significativo (Bonferroni)': p_ttest < system.config.ALPHA_BONFERRONI
        })

print("\n📋 Matriz de p-valores — Prueba t-pareada (Accuracy, 5 folds):")
display(ttest_matrix.round(4))
print("\n📋 Matriz de p-valores — Wilcoxon (Accuracy, 5 folds):")
display(wilcoxon_matrix.round(4))

pairs_summary_df = pd.DataFrame(pairs_summary)
display(pairs_summary_df)

# Se guardan todos los resultados estadísticos en el sistema para el reporte final
system.stat_tests = {
    'mcnemar': mcnemar_matrix,
    'delong': delong_matrix,
    'ttest': ttest_matrix,
    'wilcoxon': wilcoxon_matrix,
    'pairs_summary': pairs_summary_df
}
print("\n✅ Resultados estadísticos guardados en system.stat_tests")


## 📑 Reportes Profesionales — Excel, PDF y Word (CORRECCIÓN #8)

Hasta ahora solo se guardaban un `.txt` y un `.json`. Se agregan tres módulos de
exportación con formato profesional, todos guardados en `Config.REPORTS_PATH`:

- **Excel (`openpyxl`)** — `resultados_modelos.xlsx` con 4 hojas:
  `1_EDA_Stats`, `2_Model_Metrics`, `3_CV_Results`, `4_P_Values`.
- **PDF (`matplotlib.backends.backend_pdf.PdfPages`)** — `reporte_figuras.pdf`
  con 6 páginas de figuras (EDA, ROC comparativa, matrices de confusión,
  barras de Accuracy/AUC con error, heatmap de correlación, importancia de
  features).
- **Word (`python-docx`)** — `reporte_interpretativo.docx` con tablas
  formateadas y párrafos de interpretación automática.


In [ ]:
def generate_excel_report(system):
    print("\n📗 Generando reporte Excel...")
    wb = openpyxl.Workbook()
    header_font = Font(bold=True, color="FFFFFF")
    header_fill = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")

    def style_header(ws, n_cols):
        for col in range(1, n_cols + 1):
            cell = ws.cell(row=1, column=col)
            cell.font = header_font
            cell.fill = header_fill
            cell.alignment = Alignment(horizontal='center')
            ws.column_dimensions[get_column_letter(col)].width = 20

    # ---------- Hoja 1: EDA_Stats (media y std de las features de Wisconsin) ----------
    ws1 = wb.active
    ws1.title = "1_EDA_Stats"
    X, y = system._get_wisconsin_xy()
    eda_stats = pd.DataFrame({
        'Feature': X.columns,
        'Media': X.mean().values,
        'Desv. Estándar': X.std().values,
        'Mínimo': X.min().values,
        'Máximo': X.max().values,
    })
    for r in openpyxl_dataframe_to_rows(eda_stats):
        ws1.append(r)
    style_header(ws1, eda_stats.shape[1])

    # ---------- Hoja 2: Model_Metrics ----------
    ws2 = wb.create_sheet("2_Model_Metrics")
    rows = []
    for name, res in system.results.items():
        rows.append({
            'Modelo': name,
            'Accuracy': res.get('accuracy', res.get('test_accuracy', np.nan)),
            'Precision': res.get('precision', np.nan),
            'Recall': res.get('recall', np.nan),
            'F1-Score': res.get('f1', np.nan),
            'AUC': res.get('auc', np.nan),
        })
    metrics_df = pd.DataFrame(rows)
    for r in openpyxl_dataframe_to_rows(metrics_df):
        ws2.append(r)
    style_header(ws2, metrics_df.shape[1])

    # ---------- Hoja 3: CV_Results (media y std por fold, por modelo) ----------
    ws3 = wb.create_sheet("3_CV_Results")
    cv_rows = []
    if hasattr(system, 'cv_results'):
        for name, metrics in system.cv_results.items():
            cv_rows.append({
                'Modelo': name,
                'Accuracy (media)': np.mean(metrics['accuracy']),
                'Accuracy (std)': np.std(metrics['accuracy']),
                'AUC (media)': np.mean(metrics['auc']),
                'AUC (std)': np.std(metrics['auc']),
                'F1 (media)': np.mean(metrics['f1']),
                'F1 (std)': np.std(metrics['f1']),
            })
    cv_df = pd.DataFrame(cv_rows) if cv_rows else pd.DataFrame([{'Info': 'CV no ejecutada'}])
    for r in openpyxl_dataframe_to_rows(cv_df):
        ws3.append(r)
    style_header(ws3, cv_df.shape[1])

    # ---------- Hoja 4: P_Values (McNemar + t-pareada) ----------
    ws4 = wb.create_sheet("4_P_Values")
    if hasattr(system, 'stat_tests'):
        ws4.append(["Matriz de p-valores — McNemar (Accuracy, test fijo)"])
        for r in openpyxl_dataframe_to_rows(system.stat_tests['mcnemar'].reset_index().rename(columns={'index': 'Modelo'})):
            ws4.append(r)
        ws4.append([])
        ws4.append(["Matriz de p-valores — DeLong (AUC, test fijo)"])
        for r in openpyxl_dataframe_to_rows(system.stat_tests['delong'].reset_index().rename(columns={'index': 'Modelo'})):
            ws4.append(r)
        ws4.append([])
        ws4.append(["Matriz de p-valores — t-pareada (Accuracy, 5-Fold CV)"])
        for r in openpyxl_dataframe_to_rows(system.stat_tests['ttest'].reset_index().rename(columns={'index': 'Modelo'})):
            ws4.append(r)
    else:
        ws4.append(["Pruebas estadísticas no ejecutadas todavía"])

    excel_path = Path(system.config.REPORTS_PATH) / system.config.EXCEL_NAME
    wb.save(excel_path)
    print(f"✅ Reporte Excel guardado en: {excel_path}")
    return excel_path


def openpyxl_dataframe_to_rows(df):
    """Convierte un DataFrame en filas (encabezado + datos) para openpyxl."""
    rows = [list(df.columns)]
    for _, row in df.iterrows():
        rows.append([str(v) if isinstance(v, (list, dict)) else v for v in row.tolist()])
    return rows


excel_report_path = generate_excel_report(system)


In [ ]:
def generate_pdf_report(system):
    print("\n📕 Generando reporte PDF (6 páginas)...")
    pdf_path = Path(system.config.REPORTS_PATH) / system.config.PDF_NAME
    X, y = system._get_wisconsin_xy()

    with PdfPages(pdf_path) as pdf:
        # ---- Página 1: EDA ----
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        y.value_counts().rename({0: 'Benigno', 1: 'Maligno'}).plot(kind='bar', ax=axes[0], color=['#4C72B0', '#DD8452'])
        axes[0].set_title('Distribución de clases — Wisconsin')
        axes[0].set_ylabel('Casos')
        if hasattr(system, 'image_df') and not system.image_df.empty:
            system.image_df['label'].value_counts().rename({0: 'Benigno', 1: 'Maligno'}).plot(
                kind='bar', ax=axes[1], color=['#55A868', '#C44E52'])
            axes[1].set_title('Distribución de clases — Imágenes CBIS-DDSM')
            axes[1].set_ylabel('Casos')
        plt.suptitle('Página 1 — Análisis Exploratorio (EDA)', fontweight='bold')
        plt.tight_layout()
        pdf.savefig(fig); plt.close(fig)

        # ---- Página 2: Curvas ROC comparativas ----
        fig, ax = plt.subplots(figsize=(9, 8))
        for name, preds in system.predictions_test.items():
            fpr, tpr, _ = roc_curve(preds['y_true'], preds['y_proba'])
            auc_val = system.results[name].get('auc', np.nan)
            ax.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.3f})")
        ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Azar')
        ax.set_xlabel('Falsos Positivos (FPR)')
        ax.set_ylabel('Verdaderos Positivos (TPR)')
        ax.set_title('Página 2 — Curvas ROC comparativas (set de prueba fijo)', fontweight='bold')
        ax.legend(loc='lower right')
        pdf.savefig(fig); plt.close(fig)

        # ---- Página 3: Matrices de confusión (grid) ----
        n_models = len(system.results)
        n_cols = 3
        n_rows = math.ceil(n_models / n_cols)
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4.5 * n_rows))
        axes = np.array(axes).reshape(-1)
        for ax, (name, res) in zip(axes, system.results.items()):
            cm = np.array(res.get('confusion_matrix', [[0, 0], [0, 0]]))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                        xticklabels=['Benigno', 'Maligno'], yticklabels=['Benigno', 'Maligno'])
            ax.set_title(name, fontsize=11)
            ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
        for ax in axes[n_models:]:
            ax.axis('off')
        plt.suptitle('Página 3 — Matrices de Confusión (5 modelos)', fontweight='bold', y=1.02)
        plt.tight_layout()
        pdf.savefig(fig); plt.close(fig)

        # ---- Página 4: Accuracy/AUC comparativo con barras de error (std CV) ----
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        if hasattr(system, 'cv_results') and system.cv_results:
            names = list(system.cv_results.keys())
            for ax, metric_key, metric_label in zip(axes, ['accuracy', 'auc'], ['Accuracy', 'AUC']):
                means = [np.mean(system.cv_results[n][metric_key]) for n in names]
                stds = [np.std(system.cv_results[n][metric_key]) for n in names]
                ax.bar(names, means, yerr=stds, capsize=6, color=sns.color_palette('husl', len(names)))
                ax.set_title(f'{metric_label} (media ± std, 5-Fold CV)')
                ax.tick_params(axis='x', rotation=30)
        plt.suptitle('Página 4 — Comparativa con barras de error (CV)', fontweight='bold')
        plt.tight_layout()
        pdf.savefig(fig); plt.close(fig)

        # ---- Página 5: Heatmap de correlación (Wisconsin) ----
        fig, ax = plt.subplots(figsize=(14, 12))
        corr = X.corr()
        sns.heatmap(corr, cmap='coolwarm', center=0, ax=ax, cbar_kws={'shrink': 0.7})
        ax.set_title('Página 5 — Mapa de calor de correlación (features Wisconsin)', fontweight='bold')
        plt.tight_layout()
        pdf.savefig(fig); plt.close(fig)

        # ---- Página 6: Importancia de features (XGBoost tabular) ----
        fig, ax = plt.subplots(figsize=(10, 9))
        if 'tabular_xgboost' in system.models:
            importance = system.models['tabular_xgboost'].feature_importances_
            fi_df = pd.DataFrame({'Feature': X.columns, 'Importancia': importance}).sort_values('Importancia', ascending=False)
            sns.barplot(x='Importancia', y='Feature', data=fi_df, ax=ax, palette='cividis')
        ax.set_title('Página 6 — Importancia de Features (XGBoost tabular)', fontweight='bold')
        plt.tight_layout()
        pdf.savefig(fig); plt.close(fig)

    print(f"✅ Reporte PDF guardado en: {pdf_path}")
    return pdf_path


pdf_report_path = generate_pdf_report(system)


In [ ]:
def _add_df_table(doc, df, max_rows=30):
    df = df.reset_index() if df.index.name or not isinstance(df.index, pd.RangeIndex) else df
    table = doc.add_table(rows=1, cols=len(df.columns))
    table.style = 'Light Grid Accent 1'
    hdr = table.rows[0].cells
    for i, col in enumerate(df.columns):
        hdr[i].text = str(col)
        for p in hdr[i].paragraphs:
            for run in p.runs:
                run.font.bold = True
    for _, row in df.head(max_rows).iterrows():
        cells = table.add_row().cells
        for i, val in enumerate(row.tolist()):
            if isinstance(val, float):
                cells[i].text = f"{val:.3f}"
            else:
                cells[i].text = str(val)
    return table


def generate_word_report(system):
    print("\n📘 Generando reporte Word...")
    doc = Document()

    title = doc.add_heading('Reporte de Resultados — Detección de Cáncer de Mama', level=0)
    doc.add_paragraph(f"Generado automáticamente el {datetime.now().strftime('%d/%m/%Y %H:%M')}")

    # ---- 1. Resumen de modelos ----
    doc.add_heading('1. Métricas por Modelo (set de prueba)', level=1)
    rows = []
    for name, res in system.results.items():
        rows.append({
            'Modelo': name,
            'Accuracy': res.get('accuracy', res.get('test_accuracy', np.nan)),
            'Precision': res.get('precision', np.nan),
            'Recall': res.get('recall', np.nan),
            'F1-Score': res.get('f1', np.nan),
            'AUC': res.get('auc', np.nan),
        })
    metrics_df = pd.DataFrame(rows)
    _add_df_table(doc, metrics_df)

    # ---- 2. Validación cruzada ----
    doc.add_heading('2. Validación Cruzada (5-Fold)', level=1)
    if hasattr(system, 'cv_results') and system.cv_results:
        cv_rows = []
        for name, m in system.cv_results.items():
            cv_rows.append({
                'Modelo': name,
                'Accuracy (media±std)': f"{np.mean(m['accuracy']):.3f} ± {np.std(m['accuracy']):.3f}",
                'AUC (media±std)': f"{np.mean(m['auc']):.3f} ± {np.std(m['auc']):.3f}",
                'F1 (media±std)': f"{np.mean(m['f1']):.3f} ± {np.std(m['f1']):.3f}",
            })
        _add_df_table(doc, pd.DataFrame(cv_rows))
    else:
        doc.add_paragraph("La validación cruzada no fue ejecutada en esta corrida.")

    # ---- 3. Pruebas estadísticas + interpretación automática ----
    doc.add_heading('3. Pruebas Estadísticas', level=1)
    if hasattr(system, 'stat_tests'):
        doc.add_paragraph(
            f"Se aplicó corrección de Bonferroni (α = {system.config.ALPHA} / "
            f"{system.config.N_COMPARISONS} = {system.config.ALPHA_BONFERRONI:.4f}) para las comparaciones "
            f"por pares entre los modelos con validación cruzada."
        )
        doc.add_heading('3.1 Test de McNemar (Accuracy, set de prueba fijo)', level=2)
        _add_df_table(doc, system.stat_tests['mcnemar'].reset_index().rename(columns={'index': 'Modelo'}))

        doc.add_heading('3.2 Test de DeLong (AUC, set de prueba fijo)', level=2)
        _add_df_table(doc, system.stat_tests['delong'].reset_index().rename(columns={'index': 'Modelo'}))

        doc.add_heading('3.3 Prueba t-pareada (Accuracy, 5-Fold CV)', level=2)
        _add_df_table(doc, system.stat_tests['ttest'].reset_index().rename(columns={'index': 'Modelo'}))

        # ---- Interpretación automática ----
        doc.add_heading('3.4 Interpretación automática', level=2)
        acc_by_model = {k: v.get('accuracy', v.get('test_accuracy', 0)) for k, v in system.results.items()}
        best_model = max(acc_by_model, key=acc_by_model.get)
        best_acc = acc_by_model[best_model]
        f1_by_model = {k: v.get('f1', np.nan) for k, v in system.results.items()}
        best_f1_model = max(f1_by_model, key=lambda k: (f1_by_model[k] if not np.isnan(f1_by_model[k]) else -1))
        best_f1 = f1_by_model[best_f1_model]

        interp = (
            f"El modelo '{best_model}' obtuvo la mayor Accuracy en el set de prueba "
            f"({best_acc:.3f}). En términos de F1-Score, el modelo con mejor equilibrio "
            f"entre precisión y sensibilidad fue '{best_f1_model}' ({best_f1:.3f}). "
        )

        pairs_df = system.stat_tests.get('pairs_summary')
        if pairs_df is not None and len(pairs_df) > 0:
            sig_pairs = pairs_df[pairs_df['Significativo (Bonferroni)']]
            if len(sig_pairs) > 0:
                interp += (
                    f"Se encontraron {len(sig_pairs)} comparaciones con diferencia estadísticamente "
                    f"significativa en Accuracy tras la corrección de Bonferroni (t-pareada sobre los "
                    f"{system.config.N_FOLDS} folds de CV). "
                )
                for _, r in sig_pairs.iterrows():
                    interp += (
                        f"'{r['Modelo A']}' ({r['Accuracy media A']:.3f}) vs. "
                        f"'{r['Modelo B']}' ({r['Accuracy media B']:.3f}), p={r['p-valor (t-pareada)']:.4f}. "
                    )
            else:
                interp += (
                    "No se encontraron diferencias estadísticamente significativas en Accuracy entre "
                    "los modelos evaluados con validación cruzada, tras la corrección de Bonferroni."
                )
        doc.add_paragraph(interp)
    else:
        doc.add_paragraph("Las pruebas estadísticas no fueron ejecutadas en esta corrida.")

    # ---- 4. Conclusiones ----
    doc.add_heading('4. Conclusiones', level=1)
    doc.add_paragraph(
        "Este reporte fue generado automáticamente a partir de los resultados del pipeline "
        "de detección de cáncer de mama (3 modelos únicos + 2 modelos híbridos), incluyendo "
        "validación cruzada de 5 folds y pruebas estadísticas (McNemar, DeLong, t-pareada / "
        "Wilcoxon con corrección de Bonferroni)."
    )

    word_path = Path(system.config.REPORTS_PATH) / system.config.WORD_NAME
    doc.save(word_path)
    print(f"✅ Reporte Word guardado en: {word_path}")
    return word_path


word_report_path = generate_word_report(system)


## ✅ Resumen final

Al terminar de ejecutar todo el notebook deberías tener:

- `Resultados/exploratory_report.txt` — reporte de la Fase 1 (EDA).
- `Resultados/pathology_distribution.png` y
  `Resultados/ejemplos_benignas_vs_malignas.png` — gráficos del EDA.
- `Resultados/muestras/{benignas,malignas}/` — muestra física de imágenes por
  clase (Fase 1).
- `Resultados/muestras/test/{benignas,malignas}/` — muestra física del set de
  prueba (Fase 2).
- `Resultados/test_set_imagenes.csv` y `Resultados/wisconsin_test_split.csv` —
  metadatos de los sets de prueba.
- `Resultados/cv_comparison_boxplots.png` y `Resultados/cv_comparison_bars.png`
  — gráficos comparativos de la validación cruzada (4 modelos).
- `Models/` — **un archivo por modelo**:
  - `cnn_efficientnet_*.keras`
  - `tabular_xgboost_*.pkl`
  - `tabular_rf_*.pkl`
  - `hybrid_cnn_rf_*.zip` (extractor + clasificador empaquetados)
  - `hybrid_cnn_xgb_*.zip` (extractor + clasificador empaquetados)
  - `best_model_*.keras` y `best_model_*.tflite` — mejor modelo global (si es la CNN).
- `Resultados/system_save_report_*.json` — reporte final con métricas de los 5
  modelos, `best_params` del tuning y el nombre del mejor modelo.
- `Reports/resultados_modelos.xlsx` — 4 hojas: EDA, métricas por modelo, CV, p-valores.
- `Reports/reporte_figuras.pdf` — 6 páginas de figuras comparativas.
- `Reports/reporte_interpretativo.docx` — reporte con tablas e interpretación automática.

### Correcciones aplicadas sobre la versión anterior
1. ✅ Extractor de features: `GlobalAveragePooling2D` (1.792 features) en vez de la capa Dense de 128.
2. ✅ Extracción de features vectorizada por lotes (`tf.data.Dataset` + `predict` en batch).
3. ✅ Híbridos entrenados con el 100% de `train_df` (sin `sample_size`).
4. ✅ Tuning de hiperparámetros: `GridSearchCV` (XGBoost) y `RandomizedSearchCV` (Random Forest).
5. ✅ 5-Fold CV para los 4 modelos comparables (XGB tab., RF tab., Hybrid RF, Hybrid XGB).
6. ✅ Pruebas estadísticas: McNemar, DeLong y t-pareada/Wilcoxon con corrección de Bonferroni.
7. ✅ Métricas completas (Accuracy, Precision, Recall, F1, AUC, matriz de confusión) para los 5 modelos.
8. ✅ Reportes profesionales en Excel, PDF y Word.

Si el entrenamiento de la CNN sigue siendo muy lento en Colab con `EPOCHS=12`,
bájalo temporalmente a 6-8 solo para confirmar que el pipeline completo corre sin
errores, y luego súbelo para la corrida final.
